# VIZIO Analytics Lab — Full Setup Notebook

This notebook creates the complete VIZIO analytics environment with:
- **2 source databases**: `VIZIO_STREAMING` (WatchFree+ channels, AVOD content, KPIs, revenue, studios) and `VIZIO_DEVICES` (TV fleet, homescreen, apps, OEM)
- **1 analytics database**: `VIZIO_ANALYTICS_LAB` with `ANALYTICS` and `AI_OBJECTS` schemas
- **1 warehouse**: `VIZIO_LAB_WH` (MEDIUM)
- **13 tables** populated with realistic synthetic data (10,230 + 190 = 10,420 total rows)

Date range: 2024-01-01 through 2025-09-30

---
## Step 1: Create Source Databases and Schemas
Create the two source databases with their respective schemas.

In [ ]:
%%sql -r step1_create_dbs
USE ROLE ACCOUNTADMIN;

-- DATABASE 1: VIZIO_STREAMING (WatchFree+ live channels, AVOD content, KPIs, revenue, studios)
CREATE DATABASE IF NOT EXISTS VIZIO_STREAMING;
CREATE SCHEMA IF NOT EXISTS VIZIO_STREAMING.WFP;

-- DATABASE 2: VIZIO_DEVICES (TV fleet, homescreen, apps, OEM)
CREATE DATABASE IF NOT EXISTS VIZIO_DEVICES;
CREATE SCHEMA IF NOT EXISTS VIZIO_DEVICES.FLEET;
CREATE SCHEMA IF NOT EXISTS VIZIO_DEVICES.ENGAGEMENT;
CREATE SCHEMA IF NOT EXISTS VIZIO_DEVICES.APPS;
CREATE SCHEMA IF NOT EXISTS VIZIO_DEVICES.OEM;

---
## Step 2: Create All 11 Source Tables
Create the table structures for streaming and device analytics.

In [ ]:
%%sql -r step2a_streaming_tables
-- Table 1: WFP Live Channel Summary
CREATE OR REPLACE TABLE VIZIO_STREAMING.WFP.AGG_WFP_LIVE_CHANNEL_SUMMARY_WITH_CHANNEL_NAMES (
  TIME_GRAIN VARCHAR(20), DATE DATE, CHANNEL_SESSION_LENGTH NUMBER(18,2),
  OS_VERSION VARCHAR(100), OEM_BRAND VARCHAR(100), BACKEND_APP_TYPE VARCHAR(100),
  CHANNEL_TYPE VARCHAR(100), AIRINGS_KEY VARCHAR(200), EPG_CHANNEL_NAME VARCHAR(500),
  IS_OTA BOOLEAN, EPG_CHANNEL_CATEGORY VARCHAR(200), EPG_CHANNEL_CATEGORY_ID VARCHAR(50),
  AFFILIATES VARCHAR(500)
);

-- Table 2: WFP AVOD Content Summary
CREATE OR REPLACE TABLE VIZIO_STREAMING.WFP.AGG_WFP_AVOD_CONTENT_SUMMARY_WITH_METADATA_V2 (
  TIME_GRAIN VARCHAR(20), DATE DATE, PLAYER_TYPE VARCHAR(100), OS_VERSION VARCHAR(100),
  MEDIA_ID VARCHAR(100), CONTENT_TITLE VARCHAR(1000), EPISODE_TITLE VARCHAR(1000),
  SEASON_EPISODE VARCHAR(100), PRODUCTION_TYPE VARCHAR(100), GENRE VARCHAR(200),
  STUDIO VARCHAR(500), SERIES_NAME VARCHAR(500), PUBLISH_DATE DATE, RATING VARCHAR(50),
  LANGUAGE VARCHAR(50), DEVICES NUMBER(18,0), DURATION_MINS NUMBER(18,2),
  AVOD_SESSIONS NUMBER(18,0), SESSION_DURATION_PER_AVOD_SESSION NUMBER(18,2),
  SESSION_DURATION_PER_TOKEN NUMBER(18,2), LAUNCHES NUMBER(18,0), SESSION_LENGTH NUMBER(18,2),
  OEM_BRAND VARCHAR(100), BACKEND_APP_TYPE VARCHAR(100)
);

-- Table 3: WFP Global KPIs Rolling 30 Day
CREATE OR REPLACE TABLE VIZIO_STREAMING.WFP.AGG_WFP_GLOBAL_KPIS_ROLLING_30_DAY (
  DATE DATE, WFP_TOKENS NUMBER(18,0), AVOD_TOKENS NUMBER(18,0),
  WFP_AND_AVOD_TOKENS NUMBER(18,0), WFP_SESSIONS NUMBER(18,0), AVOD_SESSIONS NUMBER(18,0),
  WFP_AND_AVOD_SESSIONS NUMBER(18,0), WFP_VIEWING_HOURS NUMBER(18,2),
  WFP_APP_HOURS NUMBER(18,2), AVOD_VIEWING_HOURS NUMBER(18,2),
  WFP_AND_AVOD_VIEWING_HOURS NUMBER(18,2), WFP_AND_AVOD_APP_HOURS NUMBER(18,2),
  BACKEND_APP_TYPE VARCHAR(100)
);

-- Table 4: WFP Revenue Rolling 30 Days
CREATE OR REPLACE TABLE VIZIO_STREAMING.WFP.AGG_WFP_LIVE_AND_AVOD_REVENUE_ROLLING_30_DAYS (
  EVENT_DATE DATE, WFP_LIVE_REVENUE NUMBER(18,2), WFP_AVOD_REVENUE NUMBER(18,2)
);

-- Table 5: WFP AVOD Studio Summary
CREATE OR REPLACE TABLE VIZIO_STREAMING.WFP.AGG_WFP_AVOD_STUDIO_SUMMARY (
  TIME_GRAIN VARCHAR(20), DATE DATE, STUDIO VARCHAR(500), DEVICES NUMBER(18,0),
  AVOD_SESSIONS NUMBER(18,0), DURATION_MINS NUMBER(18,2), LAUNCHES NUMBER(18,0)
);

In [ ]:
%%sql -r step2b_device_tables
-- Table 6: TV Attributes Current
CREATE OR REPLACE TABLE VIZIO_DEVICES.FLEET.AGG_TV_ATTRIBUTE_CURRENT_MV (
  ESN VARCHAR(100), DIID VARCHAR(100), SOC VARCHAR(50), CHIPSET VARCHAR(50),
  MODEL_NAME VARCHAR(200), RJ45_MAC VARCHAR(50), FIRMWARE VARCHAR(200), SERIES VARCHAR(100),
  SIZE VARCHAR(20), TOKEN VARCHAR(200), OEM_BRAND VARCHAR(100), VIEWING VARCHAR(50),
  IP_ADDRESS VARCHAR(50), VERSION_SCPL VARCHAR(100), CONFIG VARCHAR(200),
  BINARY_ACR VARCHAR(20), BINARY_AIRPLAY VARCHAR(20), BINARY_APPLETV VARCHAR(20),
  BINARY_BLUETOOTH VARCHAR(20), BINARY_COBALT VARCHAR(20), BINARY_CONJURE VARCHAR(20),
  BINARY_DAI VARCHAR(20), BINARY_VIZIOOS VARCHAR(20), BINARY_VIZIONDK VARCHAR(20),
  BINARY_SCPL VARCHAR(20), RETAILER_CODE VARCHAR(50), RETAILER_NAME VARCHAR(200),
  DNDELIVERYCUSTOMERCOMPLETED VARCHAR(50), OS_VERSION VARCHAR(100)
);

-- Table 7: Homescreen Summary with TV Attributes
CREATE OR REPLACE TABLE VIZIO_DEVICES.ENGAGEMENT.AGG_HOMESCREEN_SUMMARY_WITH_TV_ATTRIBUTES (
  TIME_GRAIN VARCHAR(20), DATE DATE, CLIENTTYPE VARCHAR(100), SOC VARCHAR(50),
  CHIPSET VARCHAR(50), MODEL_NAME VARCHAR(200), SERIES VARCHAR(100), SIZE VARCHAR(20),
  LAUNCH_YEAR INTEGER, DEVICES NUMBER(18,0), VIEWING_MINUTES NUMBER(18,2),
  AVG_VIEWING_MINUTES NUMBER(18,2), MEDIAN_VIEWING_MINUTES NUMBER(18,2),
  OEM_BRAND VARCHAR(100)
);

-- Table 8: Homescreen NRC (New/Recurring/Churn) with TV Attributes
CREATE OR REPLACE TABLE VIZIO_DEVICES.ENGAGEMENT.AGG_HOMESCREEN_NRC_WITH_TV_ATTRIBUTES (
  MONTH DATE, SOC VARCHAR(50), CHIPSET VARCHAR(50), MODEL_NAME VARCHAR(200),
  SERIES VARCHAR(100), SIZE VARCHAR(20), LAUNCH_YEAR INTEGER,
  DISPLAY_TECHNOLOGY VARCHAR(100), ACTIVE_DEVICES NUMBER(18,0), NEW_DEVICES NUMBER(18,0),
  RECURRING_DEVICES NUMBER(18,0), RETURNING_DEVICES NUMBER(18,0),
  CHURN_1M_DEVICES NUMBER(18,0), CHURN_6M_DEVICES NUMBER(18,0),
  ACTIVE_VIEWING_MINUTES NUMBER(18,2), NEW_VIEWING_MINUTES NUMBER(18,2),
  RECURRING_VIEWING_MINUTES NUMBER(18,2), RETURNING_VIEWING_MINUTES NUMBER(18,2),
  PREVIOUS_MONTH_ACTIVE NUMBER(18,0), GROSS_CHURN_RATE1M NUMBER(18,6),
  RETURN_RATE NUMBER(18,6), NET_CHURN1M NUMBER(18,6), OEM_BRAND VARCHAR(100)
);

-- Table 9: App Devices Launches Sources
CREATE OR REPLACE TABLE VIZIO_DEVICES.APPS.AGG_APP_DEVICES_LAUNCHES_SOURCES (
  TIME_GRAIN VARCHAR(20), ACTIVITY_DATE DATE, CLIENTTYPE VARCHAR(100),
  SOURCE VARCHAR(200), APP VARCHAR(500), SOC VARCHAR(50), CHIPSET VARCHAR(50),
  MODEL_NAME VARCHAR(200), SERIES VARCHAR(100), SIZE VARCHAR(20), VOD_TYPE VARCHAR(100),
  MONETIZATION_TYPE VARCHAR(100), DEVICES NUMBER(18,0), LAUNCHES NUMBER(18,0),
  OEM_BRAND VARCHAR(100)
);

-- Table 10: OEM Launch Devices Cumulative
CREATE OR REPLACE TABLE VIZIO_DEVICES.OEM.AGG_OEM_LAUNCH_DEVICES_CUMULATIVE (
  ACTIVITY_DATE DATE, OEM_BRAND VARCHAR(100), SCTV_NEW_USERS NUMBER(18,0),
  WFP_NEW_USERS NUMBER(18,0), AVOD_NEW_USERS NUMBER(18,0),
  AUDIT_UPDATE_TS TIMESTAMP_NTZ, MODEL VARCHAR(200)
);

-- Table 11: App Metadata (Lookup)
CREATE OR REPLACE TABLE VIZIO_DEVICES.APPS.MAPPING_APP_METADATA (
  CURATED_APP_NAME VARCHAR(500), ALIAS_APP_NAME VARCHAR(500), APP_ID VARCHAR(100),
  NAME_SPACE VARCHAR(200), VOD_TYPE VARCHAR(100), MONETIZATION_TYPE VARCHAR(100),
  DEPRECATED BOOLEAN, IR_CODE VARCHAR(50), CURRENT_ROW BOOLEAN, LAUNCH_DATE DATE,
  EFFECTIVE_DATE DATE, END_DATE DATE
);

---
## Step 3: Grant Public Access
Grant usage and select permissions to the PUBLIC role on both databases.

In [ ]:
%%sql -r step3_grants
GRANT USAGE ON DATABASE VIZIO_STREAMING TO ROLE PUBLIC;
GRANT USAGE ON ALL SCHEMAS IN DATABASE VIZIO_STREAMING TO ROLE PUBLIC;
GRANT SELECT ON ALL TABLES IN DATABASE VIZIO_STREAMING TO ROLE PUBLIC;

GRANT USAGE ON DATABASE VIZIO_DEVICES TO ROLE PUBLIC;
GRANT USAGE ON ALL SCHEMAS IN DATABASE VIZIO_DEVICES TO ROLE PUBLIC;
GRANT SELECT ON ALL TABLES IN DATABASE VIZIO_DEVICES TO ROLE PUBLIC;

---
## Step 4: Create Analytics Lab Database and Warehouse
Create the analytics workspace database with schemas and a dedicated warehouse.

In [ ]:
%%sql -r step4_analytics_lab
CREATE DATABASE IF NOT EXISTS VIZIO_ANALYTICS_LAB;
CREATE SCHEMA IF NOT EXISTS VIZIO_ANALYTICS_LAB.ANALYTICS;
CREATE SCHEMA IF NOT EXISTS VIZIO_ANALYTICS_LAB.AI_OBJECTS;

CREATE WAREHOUSE IF NOT EXISTS VIZIO_LAB_WH
  WAREHOUSE_SIZE = 'MEDIUM'
  AUTO_SUSPEND = 300
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE;

USE WAREHOUSE VIZIO_LAB_WH;

---
## Step 5: Populate VIZIO_STREAMING Tables
Insert synthetic data into all 5 WatchFree+ streaming tables.

In [ ]:
%%sql -r step5a_live_channels
-- Table 1: WFP Live Channel Summary (2000 rows)
-- Channels: NBC, CBS, ABC, Fox, CNN, ESPN, HGTV, Comedy Central, Food Network, Discovery, TLC, Hallmark, BET, Nickelodeon, MTV
-- OEM_BRAND: VIZIO (~80%) and ONN (~20%), TIME_GRAIN = 'daily', session length 5-180 min
INSERT INTO VIZIO_STREAMING.WFP.AGG_WFP_LIVE_CHANNEL_SUMMARY_WITH_CHANNEL_NAMES
WITH channels AS (
  SELECT column1 AS EPG_CHANNEL_NAME, column2 AS EPG_CHANNEL_CATEGORY, column3 AS EPG_CHANNEL_CATEGORY_ID, column4 AS CHANNEL_TYPE, column5 AS AFFILIATES
  FROM VALUES
    ('NBC', 'Entertainment', 'ENT01', 'broadcast', 'NBCUniversal'),
    ('CBS', 'Entertainment', 'ENT02', 'broadcast', 'Paramount Global'),
    ('ABC', 'Entertainment', 'ENT03', 'broadcast', 'Disney'),
    ('Fox', 'Entertainment', 'ENT04', 'broadcast', 'Fox Corp'),
    ('CNN', 'News', 'NEWS01', 'cable', 'Warner Bros Discovery'),
    ('ESPN', 'Sports', 'SPT01', 'cable', 'Disney/ESPN'),
    ('HGTV', 'Lifestyle', 'LIFE01', 'cable', 'Warner Bros Discovery'),
    ('Comedy Central', 'Entertainment', 'ENT05', 'cable', 'Paramount Global'),
    ('Food Network', 'Lifestyle', 'LIFE02', 'cable', 'Warner Bros Discovery'),
    ('Discovery', 'Lifestyle', 'LIFE03', 'cable', 'Warner Bros Discovery'),
    ('TLC', 'Lifestyle', 'LIFE04', 'cable', 'Warner Bros Discovery'),
    ('Hallmark', 'Entertainment', 'ENT06', 'cable', 'Hallmark Media'),
    ('BET', 'Entertainment', 'ENT07', 'cable', 'Paramount Global'),
    ('Nickelodeon', 'Kids', 'KIDS01', 'cable', 'Paramount Global'),
    ('MTV', 'Entertainment', 'ENT08', 'cable', 'Paramount Global')
),
date_series AS (
  SELECT DATEADD(day, SEQ4(), '2024-01-01'::DATE) AS dt
  FROM TABLE(GENERATOR(ROWCOUNT => 639))
  WHERE DATEADD(day, SEQ4(), '2024-01-01'::DATE) <= '2025-09-30'::DATE
),
base AS (
  SELECT d.dt, c.EPG_CHANNEL_NAME, c.EPG_CHANNEL_CATEGORY, c.EPG_CHANNEL_CATEGORY_ID,
    c.CHANNEL_TYPE, c.AFFILIATES,
    ROW_NUMBER() OVER (ORDER BY d.dt, c.EPG_CHANNEL_NAME) AS rn
  FROM date_series d CROSS JOIN channels c
)
SELECT
  'daily' AS TIME_GRAIN, dt AS DATE,
  ROUND(5 + UNIFORM(0::FLOAT, 175::FLOAT, RANDOM()), 2) AS CHANNEL_SESSION_LENGTH,
  CASE WHEN UNIFORM(0, 1, RANDOM()) = 0 THEN '11.0.0' ELSE '12.0.0' END AS OS_VERSION,
  CASE WHEN UNIFORM(0, 4, RANDOM()) = 0 THEN 'ONN' ELSE 'VIZIO' END AS OEM_BRAND,
  CASE WHEN UNIFORM(0, 2, RANDOM()) = 0 THEN 'smartcast' ELSE 'watchfree' END AS BACKEND_APP_TYPE,
  CHANNEL_TYPE,
  'AIR-' || TO_CHAR(dt, 'YYYYMMDD') || '-' || LEFT(EPG_CHANNEL_NAME, 3) AS AIRINGS_KEY,
  EPG_CHANNEL_NAME,
  CASE WHEN CHANNEL_TYPE = 'broadcast' THEN TRUE ELSE FALSE END AS IS_OTA,
  EPG_CHANNEL_CATEGORY, EPG_CHANNEL_CATEGORY_ID, AFFILIATES
FROM base
WHERE rn <= 2000;

In [ ]:
%%sql -r step5b_avod_content
-- Table 2: WFP AVOD Content Summary (1500 rows)
-- Real-sounding titles across Drama, Comedy, Action, Documentary, Kids, Horror, Thriller, Romance, Sci-Fi
-- Studios: Lionsgate, MGM, Paramount, Sony, A24, indie, Magnolia
INSERT INTO VIZIO_STREAMING.WFP.AGG_WFP_AVOD_CONTENT_SUMMARY_WITH_METADATA_V2
WITH date_series AS (
  SELECT DATEADD(day, SEQ4() * 3, '2024-01-01'::DATE) AS dt
  FROM TABLE(GENERATOR(ROWCOUNT => 220))
  WHERE DATEADD(day, SEQ4() * 3, '2024-01-01'::DATE) <= '2025-09-30'::DATE
),
titles AS (
  SELECT column1 AS CONTENT_TITLE, column2 AS GENRE, column3 AS STUDIO, column4 AS PRODUCTION_TYPE, column5 AS SERIES_NAME, column6 AS RATING
  FROM VALUES
    ('Midnight in Manhattan', 'Drama', 'Lionsgate', 'series', 'Midnight in Manhattan', 'TV-14'),
    ('The Last Frontier', 'Action', 'Paramount', 'movie', NULL, 'R'),
    ('Backyard Astronomers', 'Documentary', 'Magnolia', 'series', 'Backyard Astronomers', 'TV-G'),
    ('Love in Transit', 'Romance', 'Sony', 'movie', NULL, 'PG-13'),
    ('Code Breakers', 'Thriller', 'A24', 'series', 'Code Breakers', 'TV-MA'),
    ('Galaxy Runners', 'Sci-Fi', 'MGM', 'movie', NULL, 'PG-13'),
    ('Laugh Track Live', 'Comedy', 'indie', 'series', 'Laugh Track Live', 'TV-14'),
    ('Shadow Creek', 'Horror', 'A24', 'movie', NULL, 'R'),
    ('Tiny Explorers', 'Kids', 'Paramount', 'series', 'Tiny Explorers', 'TV-Y'),
    ('Desert Storm Chronicles', 'Action', 'Lionsgate', 'series', 'Desert Storm Chronicles', 'TV-MA'),
    ('The Perfect Recipe', 'Comedy', 'Sony', 'movie', NULL, 'PG'),
    ('Deep Blue Unknown', 'Documentary', 'Magnolia', 'movie', NULL, 'PG'),
    ('Neon Nights', 'Drama', 'A24', 'series', 'Neon Nights', 'TV-MA'),
    ('Wild Kingdom Returns', 'Documentary', 'indie', 'series', 'Wild Kingdom Returns', 'TV-G'),
    ('Final Verdict', 'Thriller', 'MGM', 'movie', NULL, 'R'),
    ('Summer Camp Chaos', 'Kids', 'Paramount', 'series', 'Summer Camp Chaos', 'TV-Y7'),
    ('Echoes of Tomorrow', 'Sci-Fi', 'Lionsgate', 'series', 'Echoes of Tomorrow', 'TV-14'),
    ('Heart of the City', 'Romance', 'Sony', 'movie', NULL, 'PG-13'),
    ('Stand Up Spotlight', 'Comedy', 'indie', 'series', 'Stand Up Spotlight', 'TV-14'),
    ('The Forgotten Trail', 'Drama', 'MGM', 'movie', NULL, 'PG-13')
),
base AS (
  SELECT d.dt, t.*, ROW_NUMBER() OVER (ORDER BY d.dt, t.CONTENT_TITLE) AS rn
  FROM date_series d CROSS JOIN titles t
)
SELECT
  'daily', dt,
  CASE WHEN UNIFORM(0,1,RANDOM())=0 THEN 'native' ELSE 'embedded' END,
  CASE WHEN UNIFORM(0,2,RANDOM())=0 THEN '11.0.0' WHEN UNIFORM(0,2,RANDOM())=1 THEN '12.0.0' ELSE '12.1.0' END,
  'MID-' || LPAD(UNIFORM(10000,99999,RANDOM())::VARCHAR, 5, '0'),
  CONTENT_TITLE,
  CASE WHEN PRODUCTION_TYPE='series' THEN CONTENT_TITLE || ' - Episode ' || UNIFORM(1,12,RANDOM())::VARCHAR ELSE NULL END,
  CASE WHEN PRODUCTION_TYPE='series' THEN 'S' || LPAD(UNIFORM(1,4,RANDOM())::VARCHAR,2,'0') || 'E' || LPAD(UNIFORM(1,12,RANDOM())::VARCHAR,2,'0') ELSE NULL END,
  PRODUCTION_TYPE, GENRE, STUDIO, SERIES_NAME,
  DATEADD(day, -UNIFORM(30,730,RANDOM()), dt),
  RATING,
  CASE WHEN UNIFORM(0,9,RANDOM())=0 THEN 'Spanish' ELSE 'English' END,
  UNIFORM(100, 50000, RANDOM())::NUMBER(18,0),
  ROUND(UNIFORM(500::FLOAT, 150000::FLOAT, RANDOM()), 2),
  UNIFORM(200, 80000, RANDOM())::NUMBER(18,0),
  ROUND(UNIFORM(3::FLOAT, 45::FLOAT, RANDOM()), 2),
  ROUND(UNIFORM(1::FLOAT, 25::FLOAT, RANDOM()), 2),
  UNIFORM(150, 60000, RANDOM())::NUMBER(18,0),
  ROUND(UNIFORM(2::FLOAT, 120::FLOAT, RANDOM()), 2),
  CASE WHEN UNIFORM(0,4,RANDOM())=0 THEN 'ONN' ELSE 'VIZIO' END,
  CASE WHEN UNIFORM(0,2,RANDOM())=0 THEN 'smartcast' ELSE 'watchfree' END
FROM base
WHERE rn <= 1500;

In [ ]:
%%sql -r step5c_global_kpis
-- Table 3: WFP Global KPIs Rolling 30 Day (600 rows)
-- WFP_TOKENS growing from ~8M to ~11M, viewing hours growing proportionally
INSERT INTO VIZIO_STREAMING.WFP.AGG_WFP_GLOBAL_KPIS_ROLLING_30_DAY
WITH date_series AS (
  SELECT DATEADD(day, SEQ4(), '2024-01-01'::DATE) AS dt
  FROM TABLE(GENERATOR(ROWCOUNT => 639))
  WHERE DATEADD(day, SEQ4(), '2024-01-01'::DATE) <= '2025-09-30'::DATE
),
app_types AS (
  SELECT column1 AS BACKEND_APP_TYPE FROM VALUES ('smartcast'), ('watchfree'), ('mobile')
),
base AS (
  SELECT d.dt, a.BACKEND_APP_TYPE,
    ROW_NUMBER() OVER (ORDER BY d.dt, a.BACKEND_APP_TYPE) AS rn
  FROM date_series d CROSS JOIN app_types a
)
SELECT
  dt AS DATE,
  ROUND(8000000 + (DATEDIFF(day, '2024-01-01', dt) * 4700) + UNIFORM(-200000::FLOAT, 200000::FLOAT, RANDOM()))::NUMBER(18,0),
  ROUND(5000000 + (DATEDIFF(day, '2024-01-01', dt) * 3100) + UNIFORM(-150000::FLOAT, 150000::FLOAT, RANDOM()))::NUMBER(18,0),
  ROUND(11000000 + (DATEDIFF(day, '2024-01-01', dt) * 6200) + UNIFORM(-300000::FLOAT, 300000::FLOAT, RANDOM()))::NUMBER(18,0),
  ROUND(3000000 + (DATEDIFF(day, '2024-01-01', dt) * 1800) + UNIFORM(-100000::FLOAT, 100000::FLOAT, RANDOM()))::NUMBER(18,0),
  ROUND(2000000 + (DATEDIFF(day, '2024-01-01', dt) * 1200) + UNIFORM(-80000::FLOAT, 80000::FLOAT, RANDOM()))::NUMBER(18,0),
  ROUND(4500000 + (DATEDIFF(day, '2024-01-01', dt) * 2800) + UNIFORM(-150000::FLOAT, 150000::FLOAT, RANDOM()))::NUMBER(18,0),
  ROUND(1200000 + (DATEDIFF(day, '2024-01-01', dt) * 700) + UNIFORM(-50000::FLOAT, 50000::FLOAT, RANDOM()), 2),
  ROUND(1500000 + (DATEDIFF(day, '2024-01-01', dt) * 850) + UNIFORM(-60000::FLOAT, 60000::FLOAT, RANDOM()), 2),
  ROUND(800000 + (DATEDIFF(day, '2024-01-01', dt) * 500) + UNIFORM(-40000::FLOAT, 40000::FLOAT, RANDOM()), 2),
  ROUND(1800000 + (DATEDIFF(day, '2024-01-01', dt) * 1050) + UNIFORM(-80000::FLOAT, 80000::FLOAT, RANDOM()), 2),
  ROUND(2200000 + (DATEDIFF(day, '2024-01-01', dt) * 1250) + UNIFORM(-90000::FLOAT, 90000::FLOAT, RANDOM()), 2),
  BACKEND_APP_TYPE
FROM base
WHERE rn <= 600;

In [ ]:
%%sql -r step5d_revenue
-- Table 4: WFP Revenue Rolling 30 Days (600 rows)
-- WFP_LIVE_REVENUE ~$50K-$120K/day, WFP_AVOD_REVENUE ~$30K-$80K/day with growth trend
INSERT INTO VIZIO_STREAMING.WFP.AGG_WFP_LIVE_AND_AVOD_REVENUE_ROLLING_30_DAYS
WITH date_series AS (
  SELECT DATEADD(day, SEQ4(), '2024-01-01'::DATE) AS dt
  FROM TABLE(GENERATOR(ROWCOUNT => 639))
  WHERE DATEADD(day, SEQ4(), '2024-01-01'::DATE) <= '2025-09-30'::DATE
)
SELECT
  dt AS EVENT_DATE,
  ROUND(50000 + (DATEDIFF(day, '2024-01-01', dt) * 50) + UNIFORM(-8000::FLOAT, 8000::FLOAT, RANDOM()), 2),
  ROUND(30000 + (DATEDIFF(day, '2024-01-01', dt) * 40) + UNIFORM(-6000::FLOAT, 6000::FLOAT, RANDOM()), 2)
FROM date_series
LIMIT 600;

In [ ]:
%%sql -r step5e_studio_summary
-- Table 5: WFP AVOD Studio Summary (500 rows)
-- Top studios weighted by performance: Paramount (1.5x), Lionsgate (1.4x), Sony (1.3x), MGM (1.2x), A24 (1.1x), indie (0.8x), Magnolia (0.7x)
INSERT INTO VIZIO_STREAMING.WFP.AGG_WFP_AVOD_STUDIO_SUMMARY
WITH date_series AS (
  SELECT DATEADD(day, SEQ4() * 7, '2024-01-01'::DATE) AS dt
  FROM TABLE(GENERATOR(ROWCOUNT => 100))
  WHERE DATEADD(day, SEQ4() * 7, '2024-01-01'::DATE) <= '2025-09-30'::DATE
),
studios AS (
  SELECT column1 AS STUDIO, column2 AS weight FROM VALUES
    ('Lionsgate', 1.4), ('MGM', 1.2), ('Paramount', 1.5), ('Sony', 1.3),
    ('A24', 1.1), ('indie', 0.8), ('Magnolia', 0.7)
),
base AS (
  SELECT d.dt, s.STUDIO, s.weight,
    ROW_NUMBER() OVER (ORDER BY d.dt, s.STUDIO) AS rn
  FROM date_series d CROSS JOIN studios s
)
SELECT
  'weekly', dt, STUDIO,
  ROUND(UNIFORM(5000::FLOAT, 80000::FLOAT, RANDOM()) * weight)::NUMBER(18,0),
  ROUND(UNIFORM(8000::FLOAT, 120000::FLOAT, RANDOM()) * weight)::NUMBER(18,0),
  ROUND(UNIFORM(20000::FLOAT, 500000::FLOAT, RANDOM()) * weight, 2),
  ROUND(UNIFORM(4000::FLOAT, 60000::FLOAT, RANDOM()) * weight)::NUMBER(18,0)
FROM base
WHERE rn <= 500;

---
## Step 6: Populate VIZIO_DEVICES Tables
Insert synthetic data into all 6 device analytics tables.

In [ ]:
%%sql -r step6a_tv_attributes
-- Table 6: TV Attributes Current (500 rows)
-- Series: V-Series, M-Series, P-Series, Quantum, OLED. Sizes: 32-75". Mix of VIZIO/ONN brands
-- Includes realistic FIRMWARE versions and BINARY_* feature flags
INSERT INTO VIZIO_DEVICES.FLEET.AGG_TV_ATTRIBUTE_CURRENT_MV
WITH series_data AS (
  SELECT column1 AS SERIES, column2 AS SOC, column3 AS CHIPSET FROM VALUES
    ('V-Series', 'MT5597', 'MediaTek'), ('M-Series', 'MT5670', 'MediaTek'),
    ('P-Series', 'MT9638', 'MediaTek'), ('Quantum', 'V810', 'Novatek'),
    ('OLED', 'LG-W2', 'LG')
),
sizes AS (
  SELECT column1 AS SIZE FROM VALUES ('32'), ('40'), ('43'), ('50'), ('55'), ('65'), ('75')
),
base AS (
  SELECT s.SERIES, s.SOC, s.CHIPSET, sz.SIZE,
    ROW_NUMBER() OVER (ORDER BY s.SERIES, sz.SIZE, RANDOM()) AS rn
  FROM series_data s CROSS JOIN sizes sz CROSS JOIN TABLE(GENERATOR(ROWCOUNT => 20))
)
SELECT
  'ESN-' || LPAD(rn::VARCHAR, 8, '0'),
  'DIID-' || LPAD(UNIFORM(100000,999999,RANDOM())::VARCHAR, 6, '0'),
  SOC, CHIPSET,
  CASE WHEN UNIFORM(0,4,RANDOM())=0 THEN 'ONN' ELSE 'VIZIO' END || ' ' || SERIES || ' ' || SIZE || '"',
  UPPER(SUBSTR(MD5(RANDOM()::VARCHAR), 1, 12)),
  CASE WHEN SOC='MT5597' THEN '6.0.' || UNIFORM(18,22,RANDOM())::VARCHAR || '.6'
       WHEN SOC='MT5670' THEN '6.1.' || UNIFORM(10,15,RANDOM())::VARCHAR || '.3'
       ELSE '7.0.' || UNIFORM(5,12,RANDOM())::VARCHAR || '.1' END,
  SERIES, SIZE,
  'TKN-' || UPPER(SUBSTR(MD5(RANDOM()::VARCHAR), 1, 16)),
  CASE WHEN UNIFORM(0,4,RANDOM())=0 THEN 'ONN' ELSE 'VIZIO' END,
  CASE WHEN UNIFORM(0,5,RANDOM())=0 THEN 'inactive' ELSE 'active' END,
  UNIFORM(10,223,RANDOM())::VARCHAR || '.' || UNIFORM(0,255,RANDOM())::VARCHAR || '.' || UNIFORM(0,255,RANDOM())::VARCHAR || '.' || UNIFORM(1,254,RANDOM())::VARCHAR,
  'scpl-' || UNIFORM(3,8,RANDOM())::VARCHAR || '.' || UNIFORM(0,9,RANDOM())::VARCHAR || '.' || UNIFORM(0,9,RANDOM())::VARCHAR,
  'config-' || SERIES || '-' || SIZE,
  CASE WHEN UNIFORM(0,3,RANDOM())>0 THEN 'TRUE' ELSE 'FALSE' END,
  CASE WHEN UNIFORM(0,4,RANDOM())>0 THEN 'TRUE' ELSE 'FALSE' END,
  CASE WHEN UNIFORM(0,3,RANDOM())>0 THEN 'TRUE' ELSE 'FALSE' END,
  CASE WHEN UNIFORM(0,2,RANDOM())>0 THEN 'TRUE' ELSE 'FALSE' END,
  CASE WHEN UNIFORM(0,3,RANDOM())>0 THEN 'TRUE' ELSE 'FALSE' END,
  CASE WHEN UNIFORM(0,4,RANDOM())>0 THEN 'TRUE' ELSE 'FALSE' END,
  CASE WHEN UNIFORM(0,3,RANDOM())>0 THEN 'TRUE' ELSE 'FALSE' END,
  CASE WHEN UNIFORM(0,2,RANDOM())>0 THEN 'TRUE' ELSE 'FALSE' END,
  CASE WHEN UNIFORM(0,4,RANDOM())>0 THEN 'TRUE' ELSE 'FALSE' END,
  CASE WHEN UNIFORM(0,3,RANDOM())>0 THEN 'TRUE' ELSE 'FALSE' END,
  CASE UNIFORM(0,4,RANDOM()) WHEN 0 THEN 'BB' WHEN 1 THEN 'WM' WHEN 2 THEN 'AMZ' WHEN 3 THEN 'TGT' ELSE 'COST' END,
  CASE UNIFORM(0,4,RANDOM()) WHEN 0 THEN 'Best Buy' WHEN 1 THEN 'Walmart' WHEN 2 THEN 'Amazon' WHEN 3 THEN 'Target' ELSE 'Costco' END,
  CASE WHEN UNIFORM(0,4,RANDOM())>0 THEN 'completed' ELSE 'pending' END,
  CASE WHEN UNIFORM(0,1,RANDOM())=0 THEN '11.0.0' ELSE '12.0.0' END
FROM base
WHERE rn <= 500;

In [ ]:
%%sql -r step6b_homescreen_summary
-- Table 7: Homescreen Summary with TV Attributes (800 rows)
-- Daily grain. VIEWING_MINUTES 10-300, higher for larger screens via size_factor multiplier
INSERT INTO VIZIO_DEVICES.ENGAGEMENT.AGG_HOMESCREEN_SUMMARY_WITH_TV_ATTRIBUTES
WITH date_series AS (
  SELECT DATEADD(day, SEQ4() * 5, '2024-01-01'::DATE) AS dt
  FROM TABLE(GENERATOR(ROWCOUNT => 140))
  WHERE DATEADD(day, SEQ4() * 5, '2024-01-01'::DATE) <= '2025-09-30'::DATE
),
tv_combos AS (
  SELECT s.column1 AS SERIES, s.column2 AS SOC, s.column3 AS CHIPSET, sz.column1 AS SIZE, sz.column2 AS size_factor
  FROM (VALUES ('V-Series','MT5597','MediaTek'),('M-Series','MT5670','MediaTek'),('P-Series','MT9638','MediaTek'),('Quantum','V810','Novatek'),('OLED','LG-W2','LG')) s
  CROSS JOIN (VALUES ('32',0.6),('43',0.8),('55',1.0),('65',1.3),('75',1.5)) sz
),
base AS (
  SELECT d.dt, t.*, ROW_NUMBER() OVER (ORDER BY d.dt, t.SERIES, t.SIZE) AS rn
  FROM date_series d CROSS JOIN tv_combos t
)
SELECT
  'daily', dt,
  CASE WHEN UNIFORM(0,2,RANDOM())=0 THEN 'smartcast_home' WHEN UNIFORM(0,2,RANDOM())=1 THEN 'smartcast_tv' ELSE 'watchfree_home' END,
  SOC, CHIPSET,
  CASE WHEN UNIFORM(0,4,RANDOM())=0 THEN 'ONN' ELSE 'VIZIO' END || ' ' || SERIES || ' ' || SIZE || '"',
  SERIES, SIZE,
  UNIFORM(2019, 2025, RANDOM()),
  UNIFORM(500, 30000, RANDOM())::NUMBER(18,0),
  ROUND((10 + UNIFORM(0::FLOAT, 290::FLOAT, RANDOM())) * size_factor, 2),
  ROUND((15 + UNIFORM(0::FLOAT, 180::FLOAT, RANDOM())) * size_factor, 2),
  ROUND((10 + UNIFORM(0::FLOAT, 120::FLOAT, RANDOM())) * size_factor, 2),
  CASE WHEN UNIFORM(0,4,RANDOM())=0 THEN 'ONN' ELSE 'VIZIO' END
FROM base
WHERE rn <= 800;

In [ ]:
%%sql -r step6c_homescreen_nrc
-- Table 8: Homescreen NRC (New/Recurring/Churn) with TV Attributes (400 rows)
-- V-Series 32" = higher churn (~0.10-0.15), P-Series 65" = lower churn (~0.02-0.04)
INSERT INTO VIZIO_DEVICES.ENGAGEMENT.AGG_HOMESCREEN_NRC_WITH_TV_ATTRIBUTES
WITH months AS (
  SELECT DATE_TRUNC('month', DATEADD(month, SEQ4(), '2024-01-01'::DATE)) AS mth
  FROM TABLE(GENERATOR(ROWCOUNT => 21))
  WHERE DATEADD(month, SEQ4(), '2024-01-01'::DATE) <= '2025-09-30'::DATE
),
tv_combos AS (
  SELECT s.column1 AS SERIES, s.column2 AS SOC, s.column3 AS CHIPSET, s.column4 AS DISPLAY_TECHNOLOGY,
         sz.column1 AS SIZE, sz.column2 AS churn_base
  FROM (VALUES
    ('V-Series','MT5597','MediaTek','LED'),
    ('M-Series','MT5670','MediaTek','QLED'),
    ('P-Series','MT9638','MediaTek','QLED'),
    ('Quantum','V810','Novatek','Quantum Dot'),
    ('OLED','LG-W2','LG','OLED')
  ) s
  CROSS JOIN (VALUES ('32',0.12),('43',0.08),('50',0.06),('55',0.05),('65',0.03),('75',0.025)) sz
),
base AS (
  SELECT m.mth, t.*,
    ROW_NUMBER() OVER (ORDER BY m.mth, t.SERIES, t.SIZE) AS rn,
    CASE
      WHEN SERIES='V-Series' AND SIZE='32' THEN churn_base + UNIFORM(0::FLOAT, 0.03::FLOAT, RANDOM())
      WHEN SERIES='P-Series' AND SIZE IN ('65','75') THEN GREATEST(0.02, churn_base - 0.01 + UNIFORM(-0.005::FLOAT, 0.005::FLOAT, RANDOM()))
      WHEN SERIES='OLED' THEN GREATEST(0.02, churn_base - 0.015 + UNIFORM(-0.005::FLOAT, 0.005::FLOAT, RANDOM()))
      ELSE churn_base + UNIFORM(-0.02::FLOAT, 0.02::FLOAT, RANDOM())
    END AS churn_rate
  FROM months m CROSS JOIN tv_combos t
)
SELECT
  mth, SOC, CHIPSET,
  CASE WHEN UNIFORM(0,4,RANDOM())=0 THEN 'ONN' ELSE 'VIZIO' END || ' ' || SERIES || ' ' || SIZE || '"',
  SERIES, SIZE,
  UNIFORM(2019, 2025, RANDOM()),
  DISPLAY_TECHNOLOGY,
  UNIFORM(5000, 80000, RANDOM())::NUMBER(18,0) AS ACTIVE_DEVICES,
  ROUND(ACTIVE_DEVICES * UNIFORM(0.05::FLOAT, 0.15::FLOAT, RANDOM()))::NUMBER(18,0) AS NEW_DEVICES,
  ROUND(ACTIVE_DEVICES * UNIFORM(0.60::FLOAT, 0.80::FLOAT, RANDOM()))::NUMBER(18,0) AS RECURRING_DEVICES,
  ROUND(ACTIVE_DEVICES * UNIFORM(0.02::FLOAT, 0.08::FLOAT, RANDOM()))::NUMBER(18,0) AS RETURNING_DEVICES,
  ROUND(ACTIVE_DEVICES * churn_rate)::NUMBER(18,0) AS CHURN_1M_DEVICES,
  ROUND(ACTIVE_DEVICES * churn_rate * UNIFORM(2::FLOAT, 3.5::FLOAT, RANDOM()))::NUMBER(18,0),
  ROUND(ACTIVE_DEVICES * UNIFORM(40::FLOAT, 180::FLOAT, RANDOM()), 2),
  ROUND(NEW_DEVICES * UNIFORM(20::FLOAT, 90::FLOAT, RANDOM()), 2),
  ROUND(RECURRING_DEVICES * UNIFORM(50::FLOAT, 200::FLOAT, RANDOM()), 2),
  ROUND(RETURNING_DEVICES * UNIFORM(25::FLOAT, 100::FLOAT, RANDOM()), 2),
  ROUND(ACTIVE_DEVICES * UNIFORM(0.95::FLOAT, 1.05::FLOAT, RANDOM()))::NUMBER(18,0),
  ROUND(churn_rate, 6),
  ROUND(RETURNING_DEVICES / NULLIF(CHURN_1M_DEVICES, 0), 6),
  ROUND(churn_rate - (RETURNING_DEVICES / NULLIF(ACTIVE_DEVICES, 0)), 6),
  CASE WHEN UNIFORM(0,4,RANDOM())=0 THEN 'ONN' ELSE 'VIZIO' END
FROM base
WHERE rn <= 400;

In [ ]:
%%sql -r step6d_app_launches
-- Table 9: App Devices Launches Sources (3000 rows)
-- Apps: Netflix(3x), YouTube(2.8x), Disney+(1.8x), etc. Sources: homescreen, voice, search, deep_link, cast
INSERT INTO VIZIO_DEVICES.APPS.AGG_APP_DEVICES_LAUNCHES_SOURCES
WITH date_series AS (
  SELECT DATEADD(day, SEQ4() * 2, '2024-01-01'::DATE) AS dt
  FROM TABLE(GENERATOR(ROWCOUNT => 330))
  WHERE DATEADD(day, SEQ4() * 2, '2024-01-01'::DATE) <= '2025-09-30'::DATE
),
apps AS (
  SELECT column1 AS APP, column2 AS weight FROM VALUES
    ('Netflix', 3.0), ('Hulu', 1.2), ('Disney+', 1.8), ('Max', 1.5),
    ('Peacock', 1.0), ('Paramount+', 0.9), ('YouTube', 2.8),
    ('Apple TV+', 0.8), ('Amazon Prime Video', 1.6), ('Tubi', 1.1),
    ('Pluto TV', 0.9), ('WatchFree+', 1.3)
),
sources AS (
  SELECT column1 AS SOURCE, column2 AS src_weight FROM VALUES
    ('homescreen', 0.40), ('voice', 0.10), ('search', 0.15), ('deep_link', 0.20), ('cast', 0.15)
),
tv_data AS (
  SELECT column1 AS SERIES, column2 AS SOC, column3 AS CHIPSET, column4 AS SIZE FROM VALUES
    ('V-Series','MT5597','MediaTek','43'), ('M-Series','MT5670','MediaTek','55'),
    ('P-Series','MT9638','MediaTek','65'), ('Quantum','V810','Novatek','55')
),
base AS (
  SELECT d.dt, a.APP, a.weight, s.SOURCE, s.src_weight, t.SERIES, t.SOC, t.CHIPSET, t.SIZE,
    ROW_NUMBER() OVER (ORDER BY d.dt, a.APP, s.SOURCE) AS rn
  FROM date_series d CROSS JOIN apps a CROSS JOIN sources s CROSS JOIN tv_data t
)
SELECT
  'daily', dt,
  CASE WHEN UNIFORM(0,2,RANDOM())=0 THEN 'smartcast_home' ELSE 'smartcast_tv' END,
  SOURCE, APP, SOC, CHIPSET,
  CASE WHEN UNIFORM(0,4,RANDOM())=0 THEN 'ONN' ELSE 'VIZIO' END || ' ' || SERIES || ' ' || SIZE || '"',
  SERIES, SIZE,
  CASE WHEN APP IN ('Tubi','Pluto TV','WatchFree+') THEN 'FAST'
       WHEN APP = 'YouTube' THEN 'AVOD' ELSE 'SVOD' END,
  CASE WHEN APP IN ('Tubi','Pluto TV','WatchFree+','YouTube') THEN 'ad_supported' ELSE 'subscription' END,
  ROUND(UNIFORM(100::FLOAT, 25000::FLOAT, RANDOM()) * weight * src_weight)::NUMBER(18,0),
  ROUND(UNIFORM(200::FLOAT, 40000::FLOAT, RANDOM()) * weight * src_weight)::NUMBER(18,0),
  CASE WHEN UNIFORM(0,4,RANDOM())=0 THEN 'ONN' ELSE 'VIZIO' END
FROM base
WHERE rn <= 3000;

In [ ]:
%%sql -r step6e_oem_cumulative
-- Table 10: OEM Launch Devices Cumulative (300 rows)
-- Cumulative new users growing over time for VIZIO and ONN brands
INSERT INTO VIZIO_DEVICES.OEM.AGG_OEM_LAUNCH_DEVICES_CUMULATIVE
WITH date_series AS (
  SELECT DATEADD(day, SEQ4() * 2, '2024-01-01'::DATE) AS dt
  FROM TABLE(GENERATOR(ROWCOUNT => 330))
  WHERE DATEADD(day, SEQ4() * 2, '2024-01-01'::DATE) <= '2025-09-30'::DATE
),
brands AS (
  SELECT column1 AS OEM_BRAND, column2 AS MODEL FROM VALUES
    ('VIZIO', 'V-Series 43"'), ('VIZIO', 'M-Series 55"'), ('VIZIO', 'P-Series 65"'),
    ('ONN', 'ONN 50"'), ('ONN', 'ONN 43"')
),
base AS (
  SELECT d.dt, b.OEM_BRAND, b.MODEL,
    ROW_NUMBER() OVER (ORDER BY d.dt, b.OEM_BRAND, b.MODEL) AS rn
  FROM date_series d CROSS JOIN brands b
)
SELECT
  dt,
  OEM_BRAND,
  ROUND(50000 + DATEDIFF(day, '2024-01-01', dt) * 120 + UNIFORM(-2000::FLOAT, 2000::FLOAT, RANDOM()))::NUMBER(18,0),
  ROUND(30000 + DATEDIFF(day, '2024-01-01', dt) * 85 + UNIFORM(-1500::FLOAT, 1500::FLOAT, RANDOM()))::NUMBER(18,0),
  ROUND(20000 + DATEDIFF(day, '2024-01-01', dt) * 60 + UNIFORM(-1000::FLOAT, 1000::FLOAT, RANDOM()))::NUMBER(18,0),
  CURRENT_TIMESTAMP()::TIMESTAMP_NTZ,
  MODEL
FROM base
WHERE rn <= 300;

In [ ]:
%%sql -r step6f_app_metadata
-- Table 11: App Metadata Mapping (30 rows)
-- One row per app with VOD_TYPE (SVOD/AVOD/TVOD/FAST) and MONETIZATION_TYPE
INSERT INTO VIZIO_DEVICES.APPS.MAPPING_APP_METADATA
SELECT column1, column2, column3, column4, column5, column6, column7, column8, column9, column10::DATE, column11::DATE, column12
FROM VALUES
  ('Netflix', 'Netflix', 'APP001', 'com.netflix', 'SVOD', 'subscription', FALSE, 'IR01', TRUE, '2015-01-15', '2024-01-01', NULL),
  ('Hulu', 'Hulu', 'APP002', 'com.hulu', 'SVOD', 'subscription', FALSE, 'IR02', TRUE, '2016-03-01', '2024-01-01', NULL),
  ('Disney+', 'Disney Plus', 'APP003', 'com.disney.plus', 'SVOD', 'subscription', FALSE, 'IR03', TRUE, '2019-11-12', '2024-01-01', NULL),
  ('Max', 'HBO Max', 'APP004', 'com.wbd.max', 'SVOD', 'subscription', FALSE, 'IR04', TRUE, '2023-05-23', '2024-01-01', NULL),
  ('Peacock', 'Peacock TV', 'APP005', 'com.nbc.peacock', 'SVOD', 'subscription', FALSE, 'IR05', TRUE, '2020-07-15', '2024-01-01', NULL),
  ('Paramount+', 'Paramount Plus', 'APP006', 'com.paramount.plus', 'SVOD', 'subscription', FALSE, 'IR06', TRUE, '2021-03-04', '2024-01-01', NULL),
  ('YouTube', 'YouTube', 'APP007', 'com.google.youtube', 'AVOD', 'ad_supported', FALSE, 'IR07', TRUE, '2015-01-15', '2024-01-01', NULL),
  ('Apple TV+', 'Apple TV Plus', 'APP008', 'com.apple.tvplus', 'SVOD', 'subscription', FALSE, 'IR08', TRUE, '2019-11-01', '2024-01-01', NULL),
  ('Amazon Prime Video', 'Prime Video', 'APP009', 'com.amazon.prime', 'SVOD', 'subscription', FALSE, 'IR09', TRUE, '2015-01-15', '2024-01-01', NULL),
  ('Tubi', 'Tubi TV', 'APP010', 'com.tubi', 'AVOD', 'ad_supported', FALSE, 'IR10', TRUE, '2017-06-01', '2024-01-01', NULL),
  ('Pluto TV', 'Pluto', 'APP011', 'com.pluto.tv', 'FAST', 'ad_supported', FALSE, 'IR11', TRUE, '2016-09-01', '2024-01-01', NULL),
  ('WatchFree+', 'WatchFree Plus', 'APP012', 'com.vizio.watchfree', 'FAST', 'ad_supported', FALSE, 'IR12', TRUE, '2018-03-01', '2024-01-01', NULL),
  ('Crackle', 'Sony Crackle', 'APP013', 'com.crackle', 'AVOD', 'ad_supported', FALSE, 'IR13', TRUE, '2016-01-15', '2024-01-01', NULL),
  ('Vudu', 'Fandango at Home', 'APP014', 'com.vudu', 'TVOD', 'transactional', FALSE, 'IR14', TRUE, '2015-06-01', '2024-01-01', NULL),
  ('Plex', 'Plex', 'APP015', 'com.plex', 'AVOD', 'hybrid', FALSE, 'IR15', TRUE, '2018-01-15', '2024-01-01', NULL),
  ('Sling TV', 'Sling', 'APP016', 'com.sling', 'SVOD', 'subscription', FALSE, 'IR16', TRUE, '2017-02-01', '2024-01-01', NULL),
  ('fuboTV', 'Fubo', 'APP017', 'com.fubo', 'SVOD', 'subscription', FALSE, 'IR17', TRUE, '2018-08-01', '2024-01-01', NULL),
  ('ESPN+', 'ESPN Plus', 'APP018', 'com.espn.plus', 'SVOD', 'subscription', FALSE, 'IR18', TRUE, '2019-04-01', '2024-01-01', NULL),
  ('Discovery+', 'Discovery Plus', 'APP019', 'com.discovery.plus', 'SVOD', 'subscription', FALSE, 'IR19', TRUE, '2021-01-04', '2024-01-01', NULL),
  ('AMC+', 'AMC Plus', 'APP020', 'com.amc.plus', 'SVOD', 'subscription', FALSE, 'IR20', TRUE, '2020-10-29', '2024-01-01', NULL),
  ('BET+', 'BET Plus', 'APP021', 'com.bet.plus', 'SVOD', 'subscription', FALSE, 'IR21', TRUE, '2019-09-19', '2024-01-01', NULL),
  ('Starz', 'STARZ', 'APP022', 'com.starz', 'SVOD', 'subscription', FALSE, 'IR22', TRUE, '2017-04-01', '2024-01-01', NULL),
  ('Roku Channel', 'The Roku Channel', 'APP023', 'com.roku.channel', 'FAST', 'ad_supported', TRUE, 'IR23', TRUE, '2019-01-01', '2024-01-01', NULL),
  ('Freevee', 'Amazon Freevee', 'APP024', 'com.amazon.freevee', 'AVOD', 'ad_supported', TRUE, 'IR24', TRUE, '2020-01-15', '2023-06-01', NULL),
  ('Xumo', 'Xumo Play', 'APP025', 'com.xumo', 'FAST', 'ad_supported', FALSE, 'IR25', TRUE, '2019-05-01', '2024-01-01', NULL),
  ('Philo', 'Philo TV', 'APP026', 'com.philo', 'SVOD', 'subscription', FALSE, 'IR26', TRUE, '2018-11-01', '2024-01-01', NULL),
  ('IMDBTV', 'IMDb TV', 'APP027', 'com.imdb.tv', 'AVOD', 'ad_supported', TRUE, 'IR27', TRUE, '2019-06-01', '2022-04-01', NULL),
  ('Showtime', 'SHO', 'APP028', 'com.showtime', 'SVOD', 'subscription', TRUE, 'IR28', TRUE, '2017-03-01', '2023-06-27', NULL),
  ('MGM+', 'EPIX', 'APP029', 'com.mgm.plus', 'SVOD', 'subscription', FALSE, 'IR29', TRUE, '2022-01-15', '2024-01-01', NULL),
  ('Crunchyroll', 'Crunchyroll', 'APP030', 'com.crunchyroll', 'SVOD', 'subscription', FALSE, 'IR30', TRUE, '2018-07-01', '2024-01-01', NULL);

---
## Step 7: Create and Populate AI Tables
Create the `CUSTOMER_FEEDBACK` (150 rows of product reviews) and `VIZIO_KNOWLEDGE_BASE` (40 rows of policy/FAQ docs) tables in `VIZIO_ANALYTICS_LAB.AI_OBJECTS`.

In [ ]:
%%sql -r step7a_create_ai_tables
-- Create both AI tables
CREATE OR REPLACE TABLE VIZIO_ANALYTICS_LAB.AI_OBJECTS.CUSTOMER_FEEDBACK (
  FEEDBACK_ID NUMBER,
  FEEDBACK_DATE DATE,
  SOURCE VARCHAR(50),
  DEVICE_SERIES VARCHAR(50),
  DEVICE_SIZE VARCHAR(10),
  RATING NUMBER(1,0),
  FEEDBACK_TEXT VARCHAR(5000),
  FEEDBACK_TYPE VARCHAR(50)
);

CREATE OR REPLACE TABLE VIZIO_ANALYTICS_LAB.AI_OBJECTS.VIZIO_KNOWLEDGE_BASE (
  DOC_ID NUMBER,
  TITLE VARCHAR(500),
  CONTENT VARCHAR(10000),
  CATEGORY VARCHAR(100)
);

In [ ]:
%%sql -r fb_insert_01
-- CUSTOMER_FEEDBACK: Positive reviews / praise (rows 1-20)
INSERT INTO VIZIO_ANALYTICS_LAB.AI_OBJECTS.CUSTOMER_FEEDBACK VALUES
(1, '2024-02-14', 'amazon', 'P-Series', '65', 5, 'I cannot say enough good things about this VIZIO P-Series 65 inch TV. The picture quality is absolutely stunning, especially when watching 4K HDR content on streaming services. The Quantum Color technology really makes colors pop, and the local dimming zones provide excellent contrast for dark scenes in movies. I was particularly impressed with the WatchFree+ service that comes built in. There are hundreds of free channels available right out of the box, and the content quality is surprisingly good for a free service. The setup process was incredibly straightforward. I had the TV mounted and connected to my WiFi network within about twenty minutes. The SmartCast interface is intuitive and responsive, making it easy to switch between apps and inputs. The remote control has dedicated buttons for popular streaming services which is a nice touch. For the price point, this TV delivers performance that rivals sets costing twice as much. The gaming mode also reduces input lag significantly, making it a great choice for console gaming. Highly recommend this TV to anyone looking for premium features without the premium price tag.', 'praise'),
(2, '2024-03-22', 'bestbuy', 'M-Series', '55', 5, 'After weeks of research comparing different TV brands in the 55 inch category, I settled on the VIZIO M-Series and could not be happier with my decision. The Dolby Vision HDR support is phenomenal and really shines when watching compatible content on Disney+ and Netflix. The full array LED backlight provides uniform brightness across the entire screen with minimal blooming around bright objects. What really surprised me was the built-in WatchFree+ streaming service. I had never heard of it before but now it is one of my most used apps. The live TV channels cover everything from news to sports to entertainment, and the on-demand AVOD content library has a ton of great movies. The SmartCast operating system runs smoothly and I have not experienced any crashes or freezes in the three months I have owned this TV. The Chromecast built-in feature is also fantastic for casting content from my phone. Sound quality from the built-in speakers is decent for a flat panel, though I would still recommend a soundbar for the best experience. Overall, this is an exceptional value TV.', 'praise'),
(3, '2024-04-10', 'vizio_com', 'Quantum', '65', 5, 'Upgraded from an older VIZIO E-Series to the new Quantum model and the difference is night and day. The quantum dot technology produces incredibly vibrant and accurate colors that make everything from nature documentaries to animated movies look breathtaking. I am particularly impressed with how well this TV handles HDR10+ content. The peak brightness is excellent and really allows highlights to pop while maintaining detail in darker areas of the image. The ProGaming Engine is a great feature for my PlayStation 5 gaming sessions, with 4K 120Hz support and variable refresh rate keeping everything silky smooth. My wife loves the WatchFree+ channels, especially the cooking and home renovation channels that are available for free. We have actually reduced our cable bill because so much content is available through WatchFree+ and the other built-in streaming apps. The TV also looks great aesthetically with its slim bezel design and clean cable management.', 'praise'),
(4, '2024-05-18', 'amazon', 'V-Series', '43', 5, 'Perfect TV for my bedroom. The VIZIO V-Series 43 inch delivers excellent picture quality for the price. I was not expecting much at this price point but the colors are vivid, the contrast is good, and the viewing angles are decent enough for a bedroom setup where I am not always sitting directly in front of the screen. The smart TV features work well and I appreciate having WatchFree+ built in for those nights when I just want to channel surf without committing to a specific show. Setup was a breeze and I had it connected and streaming within fifteen minutes of opening the box. The remote is simple and functional with quick access buttons for major streaming services. This is my third VIZIO TV purchase and they continue to deliver great value. The only minor drawback is the built-in speaker quality, but I solved that with a small Bluetooth speaker. Would definitely recommend for a secondary room TV.', 'review'),
(5, '2024-06-03', 'reddit', 'OLED', '65', 5, 'Just picked up the VIZIO OLED 65 inch and I am blown away. Coming from a mid-range LED TV, the difference in picture quality is absolutely dramatic. The perfect blacks that OLED technology provides transform the viewing experience, especially for movies and prestige TV shows with dark cinematography. Watched the latest season of my favorite sci-fi series on this TV and it felt like I was seeing it for the first time. The infinite contrast ratio means you see details in shadows that were previously invisible. The color accuracy is reference-grade and the wide viewing angles mean everyone in my living room gets a great picture regardless of where they sit. The SmartCast platform is snappy on this model and WatchFree+ loads quickly with smooth channel switching. The thin profile of the OLED panel looks incredible mounted on the wall. This is genuinely the best TV I have ever owned and VIZIO deserves credit for making OLED technology accessible at this price point.', 'praise'),
(6, '2024-02-28', 'bestbuy', 'P-Series', '75', 5, 'Bought the VIZIO P-Series 75 inch for our family room and it has completely changed our movie night experience. The massive screen combined with the excellent picture quality makes it feel like having a personal cinema. The local dimming is very effective at maintaining deep blacks while keeping bright highlights punchy. Sports look fantastic on this TV with smooth motion handling and bright, colorful reproduction of grass, jerseys, and stadium lighting. My kids love using WatchFree+ to find cartoons and kids programming without needing to open a specific app. The voice control through the remote works reliably for searching content and controlling volume. I was initially worried about the size being too much for our room but at our twelve-foot viewing distance it is perfect. The TV handles reflections well in our bright living room thanks to the anti-glare coating. Very satisfied with this purchase.', 'praise'),
(7, '2024-07-15', 'amazon', 'M-Series', '50', 4, 'Solid mid-range TV from VIZIO. The M-Series 50 inch hits a sweet spot of size, features, and price for my apartment living room. Picture quality is very good for the price with nice color reproduction and adequate brightness for daytime viewing. The Dolby Vision support is a welcome feature at this price point and makes a noticeable difference with compatible content. SmartCast works well overall though it can occasionally be a bit slow to load apps compared to a dedicated streaming device. WatchFree+ has become my go-to for background TV while working from home. The selection of news channels and lifestyle content is impressive for a free service. I docked one star because the HDR performance, while decent, does not quite reach the same peaks as the P-Series I compared it against in the store. But for the price difference, the M-Series is the smarter value proposition for most people.', 'review'),
(8, '2024-03-05', 'twitter', 'V-Series', '32', 4, 'Got the VIZIO V-Series 32 inch as a kitchen TV and it is perfect for the purpose. Compact enough to fit on my countertop stand, and the picture quality is more than adequate for casual viewing while cooking. I mostly use it to watch news in the morning through WatchFree+ and stream cooking shows from YouTube. The smart features work fine though the smaller size means the interface text can be a bit hard to read from across the kitchen. WiFi connection has been stable and I have not had any issues with streaming quality. At this price point you really cannot ask for more. It does everything I need it to do without any fuss. Good little TV for a secondary space.', 'review'),
(9, '2024-08-20', 'vizio_com', 'Quantum', '55', 5, 'The VIZIO Quantum 55 inch has exceeded all my expectations. I purchased this as an upgrade from a budget brand TV and the improvement in picture quality is dramatic. The quantum dot color technology produces incredibly rich and saturated colors without looking artificial. I calibrated the TV using the built-in picture modes and the Calibrated mode is impressively accurate right out of the box. Gaming performance is excellent with low input lag and smooth motion. The VRR support works seamlessly with my Xbox Series X and I have noticed a complete elimination of screen tearing in fast-paced games. The WatchFree+ integration is seamless and having access to free live TV and on-demand content without any subscription is a huge bonus. My only wish is that VIZIO would add a few more premium channels to the WatchFree+ lineup. Overall this is an outstanding television that punches well above its weight class.', 'praise'),
(10, '2024-04-25', 'amazon', 'P-Series', '55', 5, 'This VIZIO P-Series 55 inch is the third VIZIO product I have purchased and they consistently deliver excellent quality at competitive prices. The picture processing on this model is noticeably improved over previous generations with better motion handling and upscaling of lower resolution content. I watch a lot of older shows and movies that are not available in 4K and this TV does an admirable job of making them look sharp and clean on the big screen. The full array local dimming with over a hundred zones provides excellent contrast that approaches some OLED displays in a dark room. The SmartCast ecosystem has matured nicely and the app selection covers all the major streaming services. WatchFree+ continues to add new channels regularly which keeps the free content fresh and engaging.', 'praise'),
(11, '2024-09-12', 'bestbuy', 'M-Series', '65', 4, 'Very happy with my VIZIO M-Series 65 inch purchase. Great bang for the buck in the large screen category. The picture quality is impressive for a mid-range TV with good color accuracy and adequate brightness for HDR content. The HDMI 2.1 ports are a nice future-proofing feature and work well with my PS5 for 4K gaming at higher refresh rates. WatchFree+ has become a staple in our household with the family discovering new channels every week. The cooking and travel channels are particularly popular with my wife. The TV handles fast motion in sports reasonably well though not quite at the level of the P-Series. Build quality feels solid and the slim bezels give it a premium appearance that belies its moderate price tag.', 'review'),
(12, '2024-06-30', 'reddit', 'V-Series', '50', 4, 'Picked up the VIZIO V-Series 50 inch during a summer sale and I think it is one of the best budget TVs available today. For the price, the picture quality is remarkably good with decent contrast and vibrant colors. It is not going to compete with high-end OLEDs but for a guest room or den it is more than sufficient. The SmartCast platform provides access to all the apps I need and WatchFree+ is a nice addition for free content. The remote is basic but functional and the dedicated streaming buttons are convenient. Setup was quick and painless. My only complaints are that the viewing angles could be better and the built-in speakers are somewhat tinny at higher volumes. But these are expected trade-offs at this price point and easily addressed with a soundbar.', 'review'),
(13, '2024-05-08', 'vizio_com', 'OLED', '55', 5, 'As a home theater enthusiast, I was thrilled when VIZIO entered the OLED market at their characteristically competitive price point. The OLED 55 inch does not disappoint. The pixel-level dimming produces absolute perfect blacks and the contrast is infinite. Colors are stunning with wide gamut coverage that makes HDR content truly spectacular. I calibrated the TV with my colorimeter and the results were impressive, with Delta E values under two in most picture modes right out of the box. The panel uniformity on my unit is excellent with no visible banding or dirty screen effect. The smart TV platform runs smoothly and I appreciate having WatchFree+ for casual channel surfing between my binge-watching sessions. The processor handles upscaling well and even 720p content looks respectable. This TV is a genuine bargain for the OLED picture quality you receive.', 'praise'),
(14, '2024-07-22', 'amazon', 'Quantum', '75', 5, 'This 75 inch Quantum is an absolute beast of a TV. The picture fills my wall and the quantum dot technology ensures every inch of that display looks stunning. I went back and forth between this and a competing brand but the VIZIO offered more features at a lower price. The local dimming performance is very good for an LED TV and the HDR highlights are impressively bright. We use this primarily for family movie nights and sports viewing and it excels at both. The wide color gamut makes animated movies from Pixar and Disney look absolutely magical. WatchFree+ has been a hit with my teenagers who enjoy the variety of free channels. The SmartCast interface works well on this larger screen with easy navigation and quick app loading times.', 'praise'),
(15, '2024-08-05', 'bestbuy', 'M-Series', '43', 4, 'Good quality TV for a bedroom or office setup. The VIZIO M-Series 43 inch provides premium features in a compact size that is perfect for smaller rooms. The Dolby Vision support elevates the viewing experience and the overall picture quality is excellent for this screen size. The SmartCast platform is responsive and the app selection is comprehensive. I use WatchFree+ daily for morning news and late-night entertainment. The only reason I am not giving five stars is that the HDR peak brightness could be a bit higher for daytime viewing in my sunlit bedroom. Otherwise this is a really well-rounded television that I would recommend to anyone looking for a quality small to mid-size display.', 'review');


In [ ]:
%%sql -r fb_insert_02
-- CUSTOMER_FEEDBACK: Positive reviews / praise (rows 16-40)
INSERT INTO VIZIO_ANALYTICS_LAB.AI_OBJECTS.CUSTOMER_FEEDBACK VALUES
(16, '2024-03-15', 'twitter', 'P-Series', '65', 5, 'Just set up the new VIZIO P-Series 65 and this thing is INCREDIBLE. Picture quality rivals TVs costing double. The local dimming zones create amazing contrast and the 4K detail is razor sharp. WatchFree+ has way more content than I expected. Already cut my streaming subscriptions down because the free AVOD content is legitimately good. Gaming mode is fantastic with super low input lag. My PS5 games have never looked better. The ProGaming Engine handles variable refresh rate perfectly and I notice zero tearing even in the most demanding titles. Easily the best TV purchase I have made.', 'praise'),
(17, '2024-10-01', 'amazon', 'V-Series', '55', 4, 'The VIZIO V-Series 55 inch represents excellent value in the large screen TV market. While it does not have all the premium features of the P-Series or Quantum lines, it delivers a very satisfying viewing experience for everyday use. The 4K resolution looks sharp and the color reproduction is natural and pleasing. I primarily use this for streaming services and WatchFree+ and it handles both admirably. The SmartCast interface has improved significantly over older versions and feels much snappier now. App loading times are reasonable and the system is stable with no crashes in my two months of ownership. The remote control is well-designed with a comfortable grip and intuitive button layout. This TV is ideal for budget-conscious shoppers who still want a quality large screen experience.', 'review'),
(18, '2024-11-18', 'bestbuy', 'Quantum', '50', 5, 'The 50 inch Quantum is the perfect size for my apartment and the picture quality is extraordinary. Quantum dot technology really delivers on its promise of brighter more colorful images. I watch a lot of nature documentaries and the color rendering of ocean blues, jungle greens, and sunset oranges is simply breathtaking. The TV also handles skin tones beautifully which is important for drama and comedy programming. WatchFree+ has become my default entertainment source. I love being able to flip through live channels like the old cable TV days but without any monthly fee. The mix of news, sports, and entertainment channels covers everything I need. The SmartCast system is reliable and I appreciate the regular firmware updates that VIZIO pushes to improve performance and add new features.', 'praise'),
(19, '2024-12-05', 'reddit', 'M-Series', '55', 4, 'Upgraded my living room TV from a six year old model to the VIZIO M-Series 55 and the leap in technology is impressive. Everything from resolution to HDR to smart features is lightyears ahead of what I had before. The picture is crisp with good detail in both bright and dark scenes. The Dolby Vision implementation is solid and compatible content looks noticeably better than standard HDR. I do wish the peak brightness was a bit higher for watching in my bright living room during the afternoon but curtains solve that problem. The WatchFree+ service is a pleasant surprise and has replaced a lot of my mindless scrolling through paid apps looking for something to watch. When I just want something on in the background there is always an interesting channel to tune into.', 'review'),
(20, '2025-01-10', 'vizio_com', 'P-Series', '75', 5, 'We purchased the VIZIO P-Series 75 inch as the centerpiece of our home theater renovation and it is absolutely stunning. The massive screen size combined with the excellent picture processing creates an immersive viewing experience that has replaced our desire to go to the movie theater. The full array local dimming with hundreds of zones creates deep inky blacks while maintaining bright highlights for truly impressive HDR performance. The sound output from the built-in speakers is surprisingly good for a flat panel though we paired it with a VIZIO soundbar for the complete surround sound experience. WatchFree+ has been a revelation for our family. The kids have their own favorite channels and the AVOD movie selection is constantly being updated with new titles. SmartCast runs smoothly on this model with no lag or stuttering when navigating between apps.', 'praise'),
(21, '2025-02-14', 'amazon', 'OLED', '65', 5, 'The VIZIO OLED 65 has completely transformed how I watch television. Every frame is a work of art with the perfect blacks and infinite contrast that only OLED can deliver. I primarily watch cinematic content and the difference compared to my previous LED TV is staggering. Dark scenes in thrillers and horror movies now have incredible depth and detail that was previously lost in murky grey-black tones. The color volume is outstanding and HDR highlights pop with incredible intensity against those perfect blacks. I also appreciate how well VIZIO has optimized the SmartCast platform for this panel. The interface is fluid and WatchFree+ channels look surprisingly good even though most are not high resolution. The AI-powered upscaling does a commendable job. This TV represents the best combination of OLED picture quality and value I have seen in the market.', 'praise'),
(22, '2025-03-01', 'bestbuy', 'V-Series', '43', 5, 'Perfect second TV for our guest bedroom. The V-Series 43 inch offers all the smart features you need at a price that will not break the bank. Picture quality exceeded my expectations for a budget model with good color saturation and reasonable black levels. The SmartCast platform provides access to every streaming app we use and WatchFree+ fills in the gaps with free content. Our guests consistently comment on how nice the TV is and are surprised when I tell them the price. Setup took less than fifteen minutes from unboxing to watching our first show. The included remote is simple but effective with shortcut buttons for popular services. VIZIO continues to prove that you do not need to spend a fortune to get a quality television experience.', 'praise'),
(23, '2025-01-28', 'reddit', 'Quantum', '55', 5, 'Three months with the Quantum 55 and I remain thoroughly impressed. The quantum dot display produces colors that are vivid yet natural looking. Skin tones are rendered beautifully and the wide color gamut brings out subtle shades that other TVs in this price range simply cannot reproduce. I have been particularly impressed with the gaming performance. The combination of 4K resolution, low input lag, and VRR support makes this an excellent gaming TV. My Xbox games look stunning and play smoothly without any visual artifacts. WatchFree+ has become my daily driver for news consumption and casual viewing. The live channel experience is seamless and I love that I can flip through channels without launching a separate app.', 'praise'),
(24, '2025-04-15', 'amazon', 'M-Series', '65', 4, 'The M-Series 65 delivers reliable performance and good picture quality at a reasonable price. It handles my varied viewing needs well from streaming movies in Dolby Vision to watching live sports through cable. The large screen size makes everything more engaging and the thin bezels maximize the viewing area. SmartCast has matured into a capable platform and I appreciate the regular updates. WatchFree+ is a genuinely useful free service that provides enough content variety to keep it interesting. The only area where I would like to see improvement is the motion handling during fast-paced sports which can occasionally show some judder. Otherwise this is a very competent television that covers all the bases.', 'review'),
(25, '2024-09-30', 'vizio_com', 'P-Series', '55', 5, 'This is my second P-Series purchase and VIZIO has made significant improvements in this generation. The picture processing is more refined with better handling of gradient banding and smoother motion interpolation. The local dimming algorithm has been improved and blooming around bright objects on dark backgrounds is minimal. I am a videophile who cares deeply about image quality and this TV satisfies my critical eye at a fraction of the cost of competing premium brands. The SmartCast ecosystem is comprehensive and WatchFree+ continues to expand its channel lineup. I recently noticed several new channels added that cover niche interests like classic films and international news. Very impressed with the ongoing software support.', 'praise'),
(26, '2025-05-20', 'bestbuy', 'V-Series', '65', 4, 'Got the V-Series 65 inch for our basement entertainment area and it is perfect for the space. The picture quality is solid for a budget large screen TV and movies and sports look great on it. We use it primarily for family movie nights and sporting events and the big screen makes both experiences much more enjoyable. WatchFree+ provides plenty of entertainment options during the day when nobody wants to commit to a specific movie or show. The kids love having access to free cartoon channels. The SmartCast platform handles all our streaming apps well though it is slightly slower than a dedicated streaming stick. For the price and screen size combination this TV is hard to beat.', 'review'),
(27, '2024-11-05', 'twitter', 'OLED', '55', 5, 'Speechless. The VIZIO OLED is the real deal. Perfect blacks, amazing colors, and the best picture I have ever had in my home. Gaming on this thing with HDR enabled is a transcendent experience. WatchFree+ is just bonus content on top of an already incredible TV. Cannot recommend enough for anyone who cares about picture quality. VIZIO brought OLED to a price point that makes it accessible and I am grateful. Worth every penny and then some.', 'praise'),
(28, '2025-06-12', 'amazon', 'Quantum', '43', 4, 'Nice compact TV with premium picture quality. The Quantum 43 inch brings quantum dot technology to a smaller form factor that is perfect for bedrooms and offices. Colors are noticeably more vibrant than the V-Series and M-Series models I compared in store. The brightness levels are impressive for a smaller panel and HDR content looks dynamic even in a well-lit room. SmartCast runs smoothly and WatchFree+ loads quickly with a good selection of channels. The compact size makes it versatile for various room setups. My only minor complaint is that the sound from the smaller speakers could be fuller but that is expected with a 43 inch panel. Pairing with a small soundbar completes the package nicely.', 'review'),
(29, '2025-02-28', 'bestbuy', 'M-Series', '50', 5, 'The M-Series 50 inch continues to be one of the best values in the TV market. VIZIO has found the perfect balance of features, performance, and price in this model. The picture quality is above average for the price with good brightness, accurate colors, and respectable contrast. Dolby Vision content looks particularly impressive and the TV does a good job with HDR tone mapping. I have been using WatchFree+ as my primary source of live TV content and the experience has been smooth and enjoyable. Channel switching is fast and the programming guide is well organized by category. This TV does not have any significant weaknesses and delivers a well-rounded viewing experience that should satisfy most buyers.', 'praise'),
(30, '2025-07-08', 'reddit', 'P-Series', '65', 5, 'Six months in with the P-Series 65 and it continues to impress daily. The picture quality has actually gotten better with firmware updates that have improved the local dimming algorithm and added new picture modes. I especially appreciate the Filmmaker Mode which presents movies exactly as the directors intended. The TV handles all types of content well from bright animated kids shows to dark moody dramas. WatchFree+ has expanded its library significantly since I bought the TV and now offers even more free movies and shows. The reliability has been excellent with zero crashes or issues. This is the kind of product that makes you a brand loyalist. VIZIO has earned my trust and future purchases.', 'praise'),
(31, '2024-04-02', 'amazon', 'M-Series', '55', 4, 'Solid TV that does everything well without any glaring weaknesses. The M-Series 55 provides a comfortable step up from budget TVs without the premium price of flagship models. Picture quality is clean and detailed with good upscaling of lower resolution content. The smart platform is mature and stable with all major apps available. WatchFree+ adds genuine value with its free channel lineup. The build quality feels sturdy and the aesthetic design is clean and modern. Would have liked slightly better peak brightness for HDR but the overall package is very compelling at this price point. Happy with my purchase and would recommend to friends and family.', 'review'),
(32, '2025-03-18', 'vizio_com', 'V-Series', '55', 4, 'The V-Series 55 inch is exactly what I needed for my newly finished basement. It provides a large screen viewing experience at a budget-friendly price that makes sense for a secondary viewing area. The picture quality is surprisingly good for the price point with natural colors and adequate brightness. WatchFree+ has been the primary content source on this TV and it performs well with quick channel changes and stable streaming. The remote is straightforward and the smart platform covers all the essential apps. This is not a TV that is going to win picture quality awards but it is an honest product that delivers reliable performance day after day.', 'review'),
(33, '2025-08-01', 'bestbuy', 'Quantum', '65', 5, 'The Quantum 65 is the crown jewel of our living room. The quantum dot display produces some of the most vivid and lifelike images I have seen on an LED TV. The color volume is exceptional and the brightness levels are impressive for both HDR and SDR content. Watching nature documentaries on this TV is a visual feast with colors that feel almost three-dimensional. The contrast performance is very good for an LED panel with effective local dimming that minimizes blooming. WatchFree+ rounds out the package with an impressive selection of free channels and AVOD content. We have found ourselves watching more TV since getting this set simply because everything looks so good on it.', 'praise'),
(34, '2024-08-15', 'amazon', 'V-Series', '40', 4, 'Great value TV for a small space. The 40 inch V-Series fits perfectly on my desk as a dual-purpose monitor and TV. Picture sharpness is good at desktop viewing distances and the colors are pleasant. I use it for work during the day and switch to WatchFree+ entertainment in the evening. The smart features all work as expected and the system is responsive enough for daily use. At this size and price you are getting a lot of functionality. Not the flashiest TV in the VIZIO lineup but it is a reliable workhorse that does what it needs to do.', 'review'),
(35, '2025-04-28', 'reddit', 'P-Series', '75', 5, 'Home theater dreams realized with the P-Series 75. The combination of screen size and picture quality at this price is unmatched. The TV dominates our media room wall and creates an incredibly immersive viewing experience. HDR content is spectacular with deep blacks and punchy highlights. The quantum color processing adds richness and depth to every scene. Family movie nights have become the highlight of our week. WatchFree+ is always on in the background during the day and the kids have their favorite channels memorized. This TV has genuinely improved our quality of life at home which sounds dramatic but is absolutely true.', 'praise'),
(36, '2024-06-18', 'vizio_com', 'M-Series', '43', 4, 'The M-Series 43 inch punches above its weight in every category. For a mid-range compact TV it delivers impressive picture quality with Dolby Vision support and good color accuracy. The SmartCast platform runs well and provides access to all the streaming services I use. WatchFree+ is a nice addition that I find myself using more than expected especially for catching up on news. The size is perfect for my bedroom and the picture is detailed enough to appreciate from my six-foot viewing distance. VIZIO continues to offer the best value proposition in the TV market and this model exemplifies that philosophy.', 'review'),
(37, '2025-05-05', 'twitter', 'Quantum', '55', 5, 'Cannot believe how good this Quantum 55 looks. The colors are absolutely jaw-dropping and make my old TV look washed out in comparison. Gaming in HDR is unreal. WatchFree+ has way more free content than I expected and has basically replaced my need for two streaming subscriptions. Best tech purchase of the year easily. VIZIO knocked it out of the park with this one.', 'praise'),
(38, '2025-08-22', 'amazon', 'OLED', '65', 5, 'After years of wanting an OLED TV I finally pulled the trigger on the VIZIO OLED 65 and it was worth every day of waiting. The picture quality is simply in a different league from any LED or QLED TV I have owned. The perfect per-pixel dimming creates an image with seemingly infinite depth and the colors are vibrant yet accurate. Watching prestige television shows on this panel is an event. Every scene is rendered with such precision and beauty that it adds a new dimension to the storytelling. The WatchFree+ service provides excellent supplementary content and the SmartCast platform handles all my apps without issues. The build quality is premium with a sleek thin profile that looks beautiful on the wall.', 'praise'),
(39, '2024-10-20', 'bestbuy', 'M-Series', '55', 4, 'Reliable mid-range TV that hits all the right notes. Good picture quality with Dolby Vision, responsive smart platform, and the added bonus of WatchFree+ for free content. The M-Series 55 does not have the flash of the Quantum or the deep blacks of the OLED but it provides a consistently good experience across all types of content. Build quality is solid and the design is attractively minimal. Four stars because the competition in this price range is fierce but VIZIO holds its own with a compelling combination of features and performance. Would buy again.', 'review'),
(40, '2025-06-30', 'amazon', 'P-Series', '55', 5, 'Third VIZIO in the household and the P-Series 55 continues the tradition of excellent value. This one is for our master bedroom and the picture quality is outstanding for the space. The local dimming creates great contrast for late-night viewing and the motion handling keeps sports looking smooth and natural. WatchFree+ has become our default source for casual viewing. The channel variety has expanded significantly over the past year and there is always something worth watching. SmartCast is stable and the regular firmware updates show that VIZIO is committed to improving the experience post-purchase.', 'praise');

In [ ]:
%%sql -r fb_insert_03
-- CUSTOMER_FEEDBACK: Negative reviews / complaints (rows 41-60)
INSERT INTO VIZIO_ANALYTICS_LAB.AI_OBJECTS.CUSTOMER_FEEDBACK VALUES
(41, '2024-02-20', 'amazon', 'V-Series', '43', 2, 'Extremely frustrated with the WiFi connectivity on this TV. The VIZIO V-Series 43 inch constantly drops its wireless connection, sometimes multiple times per hour. This makes streaming virtually impossible as shows buffer and stutter constantly. I have tried everything from resetting the router to changing DNS settings to moving the router closer to the TV and nothing provides a permanent fix. The TV will work fine for an hour or two and then suddenly lose connection requiring me to go into network settings and reconnect manually. WatchFree+ is particularly affected because it relies entirely on the network connection. When the WiFi drops mid-show it forces you back to the channel guide. I have had three VIZIO TVs in the past without this issue so I suspect this is a firmware problem. Very disappointing for an otherwise decent television. Hope VIZIO addresses this in a future update because the constant disconnections make this TV nearly unusable for streaming.', 'complaint'),
(42, '2024-03-18', 'bestbuy', 'M-Series', '55', 2, 'The remote control lag on this VIZIO M-Series is maddening. There is a noticeable half-second to full-second delay between pressing a button on the remote and seeing the action on screen. This applies to everything from volume changes to navigating the SmartCast menu to switching inputs. The lag makes the entire user experience feel sluggish and frustrating. Simple tasks like browsing through WatchFree+ channels become tedious when every button press takes a beat to register. I have replaced the batteries twice and even tried a factory reset but the lag persists. Coming from a TV with an instant-response remote this is a significant downgrade in user experience. The picture quality is fine but the remote lag seriously diminishes the overall experience. This should not be an issue on a mid-range TV in this price category.', 'complaint'),
(43, '2024-05-02', 'reddit', 'V-Series', '50', 1, 'Do not buy this TV if you plan to use the smart features. The SmartCast platform on my V-Series 50 crashes constantly. Apps freeze mid-stream, the home screen becomes unresponsive, and I have to physically unplug the TV to restart it at least twice a week. Netflix crashes are the most frequent but it happens with Hulu and Disney Plus as well. WatchFree+ is also affected with channels failing to load or audio going out of sync with video. I have performed multiple factory resets and installed every firmware update available but the problems persist. The picture quality is acceptable but what good is a smart TV when the smart features do not work reliably? I ended up buying a Roku stick to bypass the built-in platform entirely which defeats the purpose of having a smart TV. Extremely disappointed and I expected better quality control from VIZIO.', 'complaint'),
(44, '2024-06-15', 'amazon', 'M-Series', '65', 2, 'Terrible experience with WatchFree Plus buffering on this VIZIO M-Series 65 inch. While other streaming apps work fine, WatchFree Plus constantly buffers and stutters making it practically unwatchable. The live channels are the worst offenders with buffering interruptions every few minutes that completely ruin the viewing experience. On-demand AVOD content is slightly better but still experiences frequent pauses. I have a 200 Mbps internet connection that handles 4K streaming on every other app without issues so this is clearly a WatchFree Plus problem, not a bandwidth issue. I contacted VIZIO support and they suggested a factory reset which did not help. For a service that VIZIO promotes as a key feature of their TVs, the streaming quality of WatchFree Plus is unacceptable. The TV itself has good picture quality but the WFP experience needs serious improvement.', 'complaint'),
(45, '2024-07-28', 'bestbuy', 'V-Series', '32', 2, 'The audio quality on this VIZIO V-Series 32 inch is embarrassingly bad. The built-in speakers produce thin tinny sound with virtually no bass response. Dialogue is often hard to understand during shows and movies, especially when background music or sound effects are present. I have tried adjusting the equalizer settings and different sound modes but nothing makes a meaningful improvement. At moderate volume levels the speakers start to distort which is completely unacceptable. For a bedroom TV where you might not want to add external speakers this is a deal-breaker. The picture quality is decent for the size and price but the audio really lets the overall experience down. I ended up having to buy a soundbar which added significantly to the total cost.', 'complaint'),
(46, '2024-04-08', 'twitter', 'Quantum', '55', 2, 'My VIZIO Quantum 55 has developed a persistent image retention issue after just four months of ownership. There are visible ghost images of channel logos and navigation bars that remain on screen even when watching different content. The problem is most noticeable on lighter backgrounds and full-screen scenes. I have tried running pixel refresh cycles and varying the content I watch but the retention continues to worsen. For a TV using quantum dot technology I expected better panel quality and durability. This is not something that should happen on a TV this new. Very concerned about the long-term reliability of this panel. Contacted VIZIO support and they want me to go through a lengthy troubleshooting process before considering a warranty claim. Not happy with the product or the support experience.', 'complaint'),
(47, '2024-09-05', 'amazon', 'P-Series', '65', 2, 'Firmware updates on my VIZIO P-Series 65 have been a nightmare. The last two updates have introduced more problems than they solved. After the most recent update the TV takes nearly two minutes to boot up from a cold start and the SmartCast home screen is noticeably slower with lag when scrolling through apps. Additionally the color calibration settings I carefully adjusted were reset to defaults after the update. I also noticed that the local dimming behavior changed and now shows more visible blooming around bright objects than before the update. WatchFree Plus also seems to have more buffering issues post-update. VIZIO needs to do more thorough testing of their firmware updates before pushing them to consumers. It is incredibly frustrating to have a TV that gets worse after updates rather than better.', 'complaint'),
(48, '2024-11-20', 'vizio_com', 'V-Series', '55', 1, 'This TV has been nothing but problems since day one. The WiFi drops constantly, apps crash regularly, the remote lag is awful, and the smart platform is painfully slow. I spend more time troubleshooting this television than actually watching it. Factory resets provide temporary relief but the issues always return within a few days. The picture quality is mediocre at best with washed out colors and poor contrast that makes dark scenes look like murky messes. The viewing angles are terrible with significant color shift if you are sitting even slightly off-center. I deeply regret this purchase and wish I had spent more on a better brand. The WatchFree Plus service would be nice in theory but it barely works on this TV due to the constant buffering and app crashes. Avoid the V-Series if you value your sanity and your time.', 'complaint'),
(49, '2025-01-15', 'bestbuy', 'M-Series', '50', 2, 'The HDMI handshake issues on this VIZIO M-Series are incredibly annoying. Every time I switch inputs from my cable box to my PS5 or to my Blu-ray player the TV takes anywhere from ten to thirty seconds to recognize the new input. During this time the screen goes black and sometimes flashes before finally displaying the content. This happens every single time without fail and makes input switching a dreaded task. I have tried different HDMI cables including certified ultra high speed cables and the problem persists on all ports. Other TVs I have owned switch inputs almost instantly so this is clearly a VIZIO issue. The CEC functionality is also unreliable with devices sometimes not being recognized or controlled properly. These connectivity issues significantly detract from what is otherwise a decent television.', 'complaint'),
(50, '2025-03-22', 'reddit', 'V-Series', '43', 1, 'I have had this TV for six months and the SmartCast platform has become almost unusable. Apps take thirty to forty-five seconds to launch, the home screen stutters when scrolling, and the entire system freezes requiring a power cycle at least once a week. WatchFree Plus channels often fail to load displaying an error message instead of content. When channels do load the quality is poor with pixelation and audio sync issues. I suspect the TV does not have enough processing power or memory to handle the SmartCast platform adequately. The hardware limitations become more apparent with each software update as VIZIO adds features that the TV cannot handle smoothly. I am stuck with a smart TV that is anything but smart. Will not be purchasing VIZIO again based on this experience.', 'complaint'),
(51, '2024-08-12', 'amazon', 'Quantum', '65', 2, 'Major disappointment with the motion handling on this VIZIO Quantum 65 inch. Sports content shows noticeable motion blur and judder that makes fast-paced action difficult to watch. Football and basketball games look like they are being viewed through a vaseline-coated lens during quick camera pans. The motion smoothing options in the settings either do not help enough or introduce a distracting soap opera effect that makes everything look unnatural. For a TV in the Quantum line with its premium positioning I expected much better motion processing. My previous TV which cost half as much handled sports motion more capably. The picture quality for static content is excellent but the motion processing is a serious weakness that limits the TVs versatility.', 'complaint'),
(52, '2025-02-08', 'bestbuy', 'M-Series', '55', 2, 'The automatic brightness limiter on this TV is extremely aggressive and ruins the HDR experience. When watching HDR content the TV constantly dims the overall picture to prevent overheating of the backlight. This means bright scenes that should be impactful and dynamic end up looking subdued and dull. The ABL kicks in noticeably during snow scenes, bright daytime exteriors, and any content with large areas of white or bright color. This effectively negates much of the HDR benefit that the TV is supposed to deliver. You can see the brightness fluctuating in real-time which is distracting and pulls you out of the content. For a TV that advertises HDR support as a key feature, the aggressive brightness limiting is a misleading omission that consumers should know about before purchasing.', 'complaint'),
(53, '2025-04-18', 'twitter', 'V-Series', '32', 2, 'The power cycling issue on my V-Series 32 is driving me crazy. The TV randomly turns itself off and back on multiple times a day. Sometimes it happens during the middle of a show which is beyond frustrating. I have checked all the power settings disabled all timers and eco modes and the problem continues. The TV also occasionally turns itself on in the middle of the night which is alarming. I suspect there is a hardware defect with the power board but VIZIO support insists on remote troubleshooting first. After three calls and multiple factory resets the issue persists. A TV that cannot reliably stay on or off is fundamentally broken regardless of how good the picture quality might be.', 'complaint'),
(54, '2024-10-30', 'amazon', 'P-Series', '55', 3, 'Mixed feelings about this P-Series 55. The picture quality when it is working properly is genuinely excellent with great HDR performance and accurate colors. However the software experience lets the hardware down. SmartCast crashes about once a week requiring a restart and the app loading times are longer than they should be for a premium-tier TV. WatchFree Plus works intermittently with some sessions being smooth and others plagued by buffering. The HDMI ARC connection with my soundbar drops out occasionally requiring me to disconnect and reconnect the cable. It feels like VIZIO put their engineering budget into the panel and picture processing but cut corners on the smart TV software and connectivity. If they could fix the software reliability this would easily be a five-star TV.', 'complaint'),
(55, '2025-05-25', 'vizio_com', 'M-Series', '65', 2, 'Six months of ownership and the Bluetooth connectivity has been unreliable since day one. Pairing headphones with the TV is a frustrating process that often requires multiple attempts. Once paired the connection drops intermittently causing audio to cut out and then resume with a noticeable delay. This makes late-night viewing with Bluetooth headphones practically impossible as you cannot maintain a stable audio connection for an entire movie or show. I have tried multiple Bluetooth headphones and earbuds with the same result so the issue is clearly with the TV. My phone connects to these same devices without any problems. For a feature that is prominently listed in the TV specifications the Bluetooth implementation on this VIZIO is seriously subpar.', 'complaint'),
(56, '2025-07-10', 'reddit', 'V-Series', '50', 2, 'The upscaling quality on this V-Series 50 is noticeably poor. Lower resolution content from cable TV and older streaming titles looks soft and muddy on this TV. Fine details are smeared and text overlays appear fuzzy and hard to read. Compared to other TVs I have tested at similar price points the VIZIO V-Series upscaling engine is behind the competition. This is a significant issue because a large portion of available content is still in 1080p or lower resolution. While native 4K content looks acceptable the poor upscaling makes this TV unsuitable for viewers who watch a mix of content resolutions. The picture quality discussion for this TV should always come with the caveat that it only looks good with 4K source material.', 'complaint'),
(57, '2024-12-15', 'bestbuy', 'Quantum', '55', 3, 'The VIZIO Quantum 55 has good picture quality but the smart TV experience leaves much to be desired. The SmartCast platform is slow to boot up and navigate. Switching between apps feels sluggish compared to a dedicated streaming device. WatchFree Plus often requires multiple attempts to load channels and the app occasionally crashes back to the home screen mid-viewing. The TV also runs noticeably warm which concerns me regarding long-term reliability. I have resorted to using a Fire TV Stick for most of my streaming because the built-in platform is too unreliable and slow. The picture quality through the external streamer is identical so the smart TV features feel like wasted potential. VIZIO needs to invest more in their software platform to match their hardware quality.', 'complaint'),
(58, '2025-06-05', 'amazon', 'V-Series', '55', 2, 'Deeply disappointed with the backlighting uniformity on my V-Series 55. The TV shows visible clouding and flashlighting in the corners that is distracting during dark scenes. The edges of the screen are noticeably brighter than the center creating an uneven picture that becomes particularly apparent when watching content with dark backgrounds or letterboxed movies. I tried adjusting the backlight settings but lowering the backlight enough to minimize the uniformity issues makes the TV too dim for comfortable viewing. For a TV that is supposed to provide an enjoyable viewing experience this panel quality control issue is unacceptable. It appears that VIZIO is cutting corners on panel quality for their budget line and consumers end up paying the price.', 'complaint'),
(59, '2025-08-15', 'vizio_com', 'M-Series', '43', 2, 'The voice control on this M-Series 43 is practically useless. The microphone in the remote has trouble picking up commands from normal viewing distances. You have to hold the remote close to your mouth and speak slowly and clearly for it to register at all. Even then the speech recognition accuracy is poor with frequent misinterpretations that lead to wrong channels or search results. The voice search integration with WatchFree Plus is particularly bad and rarely returns relevant results. I was excited about the voice control feature when purchasing this TV but it has been nothing but a frustration. Compared to the voice control on my smart speaker which works flawlessly from across the room, the VIZIO voice implementation feels like it is a generation behind.', 'complaint'),
(60, '2024-07-05', 'twitter', 'P-Series', '75', 3, 'The P-Series 75 delivers stunning picture quality marred by frustrating software issues. The panel is beautiful with excellent HDR performance and vibrant colors on the massive screen. However the SmartCast platform struggles to keep up with the demands of daily use. Apps close unexpectedly, the system occasionally freezes mid-stream, and firmware updates have been hit or miss with improvements. WatchFree Plus is a great concept but the implementation on this TV is inconsistent with buffering during prime viewing hours. I love the picture and hate the software. If VIZIO could match their excellent hardware engineering with equally competent software development this would be a perfect TV. As it stands the software unreliability prevents me from giving this the five-star rating the picture quality deserves.', 'complaint');

In [ ]:
%%sql -r fb_insert_04
-- CUSTOMER_FEEDBACK: More complaints (rows 61-80)
INSERT INTO VIZIO_ANALYTICS_LAB.AI_OBJECTS.CUSTOMER_FEEDBACK VALUES
(61, '2025-01-22', 'amazon', 'V-Series', '43', 1, 'The worst TV purchase I have ever made. The V-Series 43 arrived with a dead pixel cluster in the upper right corner that is visible on light backgrounds. When I contacted VIZIO support they said it was within their acceptable dead pixel policy and would not replace it. Beyond that the WiFi drops multiple times per day, the remote has a noticeable input lag, and SmartCast crashes regularly. WatchFree Plus buffers constantly and frequently shows an error message instead of content. The picture quality itself is mediocre with dull colors and poor black levels. I have owned budget TVs from other brands that performed significantly better than this VIZIO. The only positive thing I can say is that the return process through Amazon was smooth because this TV is going back immediately.', 'complaint'),
(62, '2025-03-10', 'bestbuy', 'Quantum', '50', 2, 'Persistent audio sync issues have plagued this Quantum 50 since purchase. The sound is consistently behind the video by a fraction of a second which creates a distracting lip-sync mismatch during dialogue scenes. The issue occurs with both the built-in speakers and when connected to my soundbar via HDMI ARC. I have tried adjusting the audio delay settings but cannot find a setting that consistently works across all inputs and apps. WatchFree Plus seems to have worse audio sync than other streaming apps. This is a fundamental issue that affects every minute of viewing and should not exist on a television at this price point. Multiple firmware updates have not addressed the problem.', 'complaint'),
(63, '2024-09-18', 'reddit', 'M-Series', '55', 2, 'The dirty screen effect on my M-Series 55 is extremely noticeable during sports viewing. When the camera pans across the field or court you can see visible banding and uneven patches across the screen that look like cloudy smudges. This is particularly apparent during hockey and soccer where the large areas of uniform color make the panel uniformity issues glaringly obvious. I tried exchanging for another unit and the replacement has the same problem just in slightly different locations. This appears to be a systemic quality control issue with the M-Series panels rather than a defective unit. For a TV marketed as good for sports this is a significant and misleading shortcoming.', 'complaint'),
(64, '2025-07-20', 'amazon', 'V-Series', '65', 2, 'The auto-brightness sensor on this V-Series 65 makes watching TV an annoying experience. The screen brightness constantly fluctuates based on the ambient light in the room creating a distracting pulsing effect. Turning off the sensor and setting brightness manually helps but then the TV is either too bright at night or too dim during the day. The lack of granular brightness scheduling or room-aware presets is a missing feature that should be standard. Additionally the TV runs warm to the touch along the top edge which concerns me. The picture quality at this size is adequate but these usability annoyances accumulate into a frustrating daily experience.', 'complaint'),
(65, '2024-05-30', 'vizio_com', 'P-Series', '55', 3, 'I have a love-hate relationship with this P-Series. The picture quality is genuinely impressive with excellent local dimming and color accuracy. However the SmartCast software is unreliable and sluggish. Apps take too long to load, the system freezes during extended viewing sessions, and WatchFree Plus buffering interrupts what should be seamless channel surfing. The gap between hardware quality and software quality on this TV is concerning. If VIZIO put the same effort into their software platform as they do into their display technology this would be the best TV in its class. Instead it is a TV with a great picture that is hampered by a frustrating user experience.', 'complaint'),
(66, '2025-09-01', 'bestbuy', 'V-Series', '32', 2, 'WiFi problems continue to plague VIZIO TVs. My V-Series 32 cannot maintain a stable wireless connection for more than thirty minutes at a time. Streaming through WatchFree Plus or any other app is a constant battle against buffering and disconnections. I have replaced my router, changed WiFi channels, and even set up a dedicated access point near the TV without improvement. The Ethernet port would solve this but routing a cable to the TV location is not practical. A smart TV that cannot reliably connect to the internet is not really smart at all. Very disappointed in the wireless performance and reliability of this product.', 'complaint'),
(67, '2024-11-08', 'twitter', 'M-Series', '50', 2, 'My M-Series 50 just died after eight months. Screen went completely black with no backlight. Audio still works but no picture at all. VIZIO warranty support is making me jump through hoops to get a replacement. Been waiting two weeks for a service appointment. Really poor experience for a TV that should last years not months. Starting to think VIZIO sacrifices reliability for low prices. Friends are warning me to go with a different brand next time and I am starting to agree with them.', 'complaint'),
(68, '2025-04-05', 'amazon', 'Quantum', '55', 2, 'The CEC implementation on this Quantum 55 is broken. Devices connected via HDMI frequently lose CEC communication causing my soundbar to stop responding to volume commands and my streaming device to not turn the TV on automatically. I have to power cycle everything to restore CEC functionality at least twice a week. This is a basic connectivity feature that should work reliably on any modern TV. The picture quality is good but these connectivity issues make daily usage more cumbersome than it needs to be.', 'complaint'),
(69, '2025-06-18', 'reddit', 'V-Series', '55', 1, 'Absolutely do not recommend this TV. In addition to the chronic WiFi dropping and SmartCast crashes that seem to affect all V-Series models, my unit has developed a high-pitched buzzing sound from the panel that is audible during quiet scenes. The buzzing changes intensity with different on-screen content suggesting it is related to the backlight or power delivery. Combined with the terrible built-in speaker quality this TV is an assault on the ears. The only way to enjoy it is with external speakers loud enough to drown out the panel noise. For a company that has been making TVs for decades this level of quality control is embarrassing.', 'complaint'),
(70, '2025-08-28', 'bestbuy', 'M-Series', '65', 2, 'Severe light bleed from the edges makes this M-Series 65 terrible for watching content with black bars. Letterboxed movies show bright spots in all four corners that are impossible to ignore once you notice them. The backlight bleed is visible even with the backlight setting reduced significantly. This is a panel quality issue that no amount of settings adjustment can fix. I specifically chose the M-Series over the V-Series expecting better panel quality and was very disappointed to find this level of light bleed on what is supposed to be a step up in the product line.', 'complaint'),
(71, '2024-08-25', 'amazon', 'V-Series', '50', 2, 'The VIZIO V-Series 50 has been plagued with overheating issues since I brought it home. The TV gets uncomfortably hot along the top edge after just an hour of use and occasionally displays a thermal warning message before automatically dimming the screen. This happens even in a well-ventilated room with plenty of clearance around the TV. The forced dimming ruins the viewing experience as the picture becomes noticeably darker and less vibrant. I should not have to worry about a TV overheating during normal use. Combined with the sluggish SmartCast performance and frequent WiFi disconnections this TV has been a thoroughly disappointing purchase.', 'complaint'),
(72, '2025-02-18', 'vizio_com', 'P-Series', '65', 3, 'The P-Series picture quality is exceptional but I am giving three stars because of persistent audio issues through HDMI eARC. My Dolby Atmos soundbar frequently loses its Atmos signal defaulting to stereo output without any visual indication on screen. I only notice when the surround sound effect disappears and I have to check the soundbar display. Reconnecting requires switching inputs back and forth or power cycling the soundbar. This happens at least once per viewing session and is incredibly disruptive especially during movies where the Atmos audio is a key part of the experience. VIZIO needs to improve their HDMI eARC implementation.', 'complaint'),
(73, '2025-05-12', 'amazon', 'V-Series', '43', 2, 'The SmartCast update that rolled out last week completely broke my V-Series. The TV now takes three to four minutes to fully boot up and reach the home screen. Once loaded the interface is painfully slow with noticeable lag on every interaction. WatchFree Plus no longer loads at all displaying a generic error message. Several streaming apps including Netflix and Hulu fail to launch on the first attempt and sometimes require three or four tries. This was not happening before the update and VIZIO has not acknowledged the issue or provided a fix. Pushing a broken update to customers is irresponsible.', 'complaint'),
(74, '2024-06-22', 'bestbuy', 'Quantum', '65', 3, 'Good picture quality let down by the most frustrating remote control experience I have ever had. The remote loses its Bluetooth connection to the TV at random requiring me to re-pair it which involves pressing a tiny button on the back with a paperclip. This happens at least once a week and sometimes multiple times in a single day. Additionally the remote buttons feel mushy and unresponsive with several requiring firm deliberate presses to register. The volume rocker is particularly problematic with inconsistent responses. A TV is only as good as its remote control interface and the VIZIO remote is a weak link.', 'complaint'),
(75, '2025-09-10', 'twitter', 'M-Series', '55', 2, 'Five months in and the screen has developed permanent burn-in of the SmartCast home bar at the bottom. I can see the outline of the navigation bar on every input and every piece of content. This should NOT happen on an LED TV. Only had the TV for five months. VIZIO support says they will send a technician but I have been waiting two weeks for a callback. Avoid the M-Series if you value long-term panel quality. This is a manufacturing defect plain and simple.', 'complaint'),
(76, '2024-04-15', 'amazon', 'V-Series', '55', 2, 'The auto-update feature on this V-Series cannot be disabled and has caused multiple problems. The TV updates firmware automatically in the middle of the night and twice now I have woken up to find the TV stuck in a boot loop requiring a factory reset that erases all my settings and app logins. There should be an option to control when updates are installed or at minimum to delay them. Having a TV reset itself without warning is unacceptable. I lose all my custom picture settings, app arrangements, and WiFi passwords every time this happens. Basic respect for the user experience demands giving consumers control over their own device updates.', 'complaint'),
(77, '2025-07-05', 'reddit', 'Quantum', '50', 2, 'The HDR implementation on the Quantum 50 is broken in my opinion. The tone mapping is way too aggressive and clips highlights while simultaneously crushing shadow detail. HDR content often looks worse than SDR on this TV because the processing strips away the nuance that HDR is supposed to preserve. Bright specular highlights are blown out to white and dark areas become impenetrable black voids. The HDR picture modes available in the settings do not adequately address this as even the most conservative mode over-processes the image. For a TV that prominently features HDR support in its marketing the actual HDR experience is deeply disappointing.', 'complaint'),
(78, '2024-12-28', 'bestbuy', 'M-Series', '43', 3, 'Average TV with some annoying quirks. Picture quality is fine for casual viewing but the smart platform is slow and the remote lag is noticeable. WatchFree Plus works about seventy percent of the time with the other thirty percent spent waiting for channels to load or dealing with buffering. The TV also makes a clicking sound when turning on and off that is louder than expected. Not a terrible TV but not one I would enthusiastically recommend either. There are better options in this price range from competitors.', 'complaint'),
(79, '2025-08-08', 'amazon', 'V-Series', '32', 1, 'Returned this TV after one week. The picture quality is terrible for a modern TV even at this price point. Colors are washed out and inaccurate, black levels are grey at best, and the viewing angles are so narrow that anyone sitting more than twenty degrees off-center sees a completely degraded image. The smart features are painfully slow with apps taking over thirty seconds to launch. WatchFree Plus buffered constantly on my high-speed internet connection. The remote feels cheap and the buttons are already starting to stick after just a few days of use. There are budget TVs from other brands that perform significantly better. Save your money and look elsewhere.', 'complaint'),
(80, '2025-03-28', 'vizio_com', 'P-Series', '75', 3, 'Conflicted about this P-Series 75. The picture is genuinely beautiful and the size is perfect for our home theater. But the software issues are real and persistent. SmartCast freezes about once a week, WatchFree Plus has regular buffering problems, and the system generally feels underpowered for the demands placed on it. I have learned to keep a streaming stick as a backup for when the built-in platform acts up. It is frustrating because the hardware deserves better software. The picture quality keeps me from rating this any lower but the software problems prevent me from rating it higher. VIZIO please invest in your software team.', 'complaint');

In [ ]:
%%sql -r fb_insert_05
-- CUSTOMER_FEEDBACK: Feature requests (rows 81-110)
INSERT INTO VIZIO_ANALYTICS_LAB.AI_OBJECTS.CUSTOMER_FEEDBACK VALUES
(81, '2024-03-12', 'reddit', 'P-Series', '65', 4, 'Love the picture quality on my P-Series 65 but I really wish VIZIO would improve the voice search functionality. The current voice search is limited to basic queries and does not understand natural language commands well. I would love to be able to say things like show me action movies from the last year or find something funny to watch and have the TV intelligently search across all installed apps and WatchFree Plus simultaneously. The current implementation only searches within one app at a time which defeats the purpose of a unified smart TV platform. A universal voice search that spans all content sources including WatchFree Plus live channels would be a game-changing feature.', 'feature_request'),
(82, '2024-05-20', 'amazon', 'M-Series', '55', 4, 'Great TV overall but I have a specific feature request for WatchFree Plus. Please add more live sports channels to the WFP lineup. Currently the sports coverage is limited and I would love to see dedicated channels for MLS, college sports, and international soccer. The existing news and entertainment channels are good but sports fans are underserved in the current WatchFree Plus offering. Also it would be great to have a sports-specific section in the WFP guide that aggregates all live and upcoming sporting events across channels.', 'feature_request'),
(83, '2024-07-08', 'vizio_com', 'Quantum', '55', 4, 'The Quantum 55 has excellent picture quality and I would love to see VIZIO add native Dolby Vision gaming support. Currently the Dolby Vision implementation works great for streaming content but is not available for gaming inputs. With the next generation of consoles and PC graphics cards supporting Dolby Vision for games this would be a meaningful addition that keeps the TV competitive. Additionally having dedicated gaming picture presets for different genres like FPS, RPG, and racing would help users optimize their display settings without manual calibration each time.', 'feature_request'),
(84, '2024-09-15', 'reddit', 'OLED', '65', 5, 'Absolutely love this OLED but please VIZIO add a proper multi-view or picture-in-picture mode. I would love to watch a game in the main window while keeping a second game or news ticker in a smaller window. This was a standard feature on TVs years ago and its absence on modern smart TVs is puzzling. The OLED panel has more than enough resolution to support a PIP mode without sacrificing image quality. Also adding a split-screen mode for two HDMI inputs simultaneously would be amazing for sports fans during tournament season.', 'feature_request'),
(85, '2024-11-02', 'amazon', 'P-Series', '75', 4, 'Feature request for WatchFree Plus. Please add parental controls specific to WFP channels and content. Currently the parental control options for WatchFree Plus are limited compared to other streaming apps on the TV. I would like the ability to block specific WFP channels, set viewing time limits for WFP specifically, and have content ratings filters that automatically block age-inappropriate content on WFP channels. With kids in the house I need more granular control over what free content they can access.', 'feature_request'),
(86, '2025-01-05', 'bestbuy', 'M-Series', '65', 3, 'The M-Series hardware is decent but the SmartCast platform needs a major overhaul. Specifically I would like to see VIZIO add customizable home screen layouts where users can arrange their favorite apps and WatchFree Plus channels in whatever order they prefer. The current fixed layout wastes valuable screen real estate on promoted content and ads rather than showing what I actually want to access. Additionally adding a favorites or watchlist feature that works across all apps and WFP would make content discovery much easier.', 'feature_request'),
(87, '2025-02-22', 'reddit', 'V-Series', '50', 3, 'I wish VIZIO would add proper Dolby Vision support to the V-Series lineup. I understand it is the budget line but Dolby Vision has become so prevalent in streaming content that not having it on any TV in 2025 feels like a significant omission. Even a basic Dolby Vision implementation that converts to HDR10 would be better than the current situation where Dolby Vision content defaults to SDR on V-Series TVs. Also please add more WatchFree Plus channels in languages other than English. There is a huge Spanish-speaking audience that would benefit from more Spanish language channels on WFP.', 'feature_request'),
(88, '2025-04-10', 'vizio_com', 'Quantum', '65', 4, 'Request for VIZIO to implement a proper ambient mode or art display feature for the Quantum line. When the TV is not actively being watched it would be great to display artwork, family photos, or subtle ambient content that turns the TV into a decorative element rather than a black rectangle on the wall. Samsung has successfully implemented this with their Frame TV concept and I believe VIZIO could offer something similar as a software update. The Quantum display has excellent color accuracy that would show artwork beautifully.', 'feature_request'),
(89, '2024-06-25', 'amazon', 'P-Series', '55', 4, 'Please add cloud DVR functionality to WatchFree Plus. The ability to record live WFP channels and watch them later would be an incredible feature that would genuinely compete with paid cable and streaming alternatives. Even a limited recording capacity of twenty hours would add significant value to the WFP platform. Currently if you miss a live show on WFP there is no way to go back and watch it unless it happens to be available in the on-demand section. Time-shifting is a fundamental TV feature that WatchFree Plus needs to remain competitive.', 'feature_request'),
(90, '2024-08-30', 'twitter', 'M-Series', '55', 4, 'VIZIO needs to add proper casting support for AirPlay 2 on all models. My M-Series 55 supports AirPlay but the implementation is inconsistent and frequently drops connection during casting from my iPhone. I would love reliable AirPlay 2 with full HomeKit integration so I can control the TV through Siri and include it in home automation routines. This is increasingly important as more people build Apple smart home ecosystems. Please VIZIO make AirPlay a first-class citizen on SmartCast.', 'feature_request'),
(91, '2025-06-15', 'reddit', 'OLED', '55', 5, 'This OLED is phenomenal but I have a request for the firmware team. Please add more granular local dimming controls and picture calibration tools accessible through the standard settings menu. Currently the calibration options are limited to basic brightness contrast and color settings. Having access to white balance controls, gamma curves, and color management system adjustments through the TV menu would allow enthusiasts to fine-tune their picture without requiring expensive calibration equipment.', 'feature_request'),
(92, '2025-03-15', 'amazon', 'V-Series', '55', 3, 'I would really appreciate it if VIZIO could add Ethernet over USB support or improve the WiFi antenna in the V-Series. The wireless connectivity issues on this line are well documented and having a USB Ethernet adapter option would give users a workaround for homes where running an Ethernet cable directly to the TV is not practical. Also adding WiFi 6 support to the V-Series would likely solve many of the connection stability issues that plague this product line.', 'feature_request'),
(93, '2024-10-12', 'vizio_com', 'P-Series', '65', 4, 'Request for better integration between WatchFree Plus and the SmartCast home screen. I would love to see personalized WFP content recommendations on the home screen based on my viewing history. Currently the WFP section on SmartCast shows generic popular content rather than personalized suggestions. Machine learning-based recommendations that learn my preferences over time would make content discovery much more efficient and would increase my engagement with WFP.', 'feature_request'),
(94, '2025-05-08', 'bestbuy', 'Quantum', '55', 4, 'Would love to see VIZIO implement a proper gaming hub interface similar to what some competitors offer. A dedicated section in SmartCast that detects connected gaming consoles, shows recently played games, optimizes picture settings per game, and provides quick access to gaming-specific features like VRR status and input lag monitoring. The current approach of manually switching to game mode is functional but a unified gaming dashboard would elevate the experience significantly.', 'feature_request'),
(95, '2024-12-20', 'reddit', 'M-Series', '50', 3, 'VIZIO please add support for USB media playback with more codecs and formats. The current USB media player on my M-Series 50 is very limited in the file formats it supports. I have a collection of movies in MKV format with various audio codecs that the TV simply cannot play. Adding support for MKV, HEVC, DTS, and high bitrate audio formats through USB playback would be a significant quality of life improvement.', 'feature_request'),
(96, '2025-07-25', 'amazon', 'V-Series', '43', 3, 'Feature request: please add a proper sleep timer that gradually reduces volume and brightness before turning off the TV. The current sleep timer just cuts the TV off abruptly which can be jarring when you are falling asleep. A gradual dimming and volume reduction over the last five minutes before shutdown would be a much more pleasant experience. Also adding smart alarm functionality where the TV can turn on in the morning with a specific WatchFree Plus channel would be useful.', 'feature_request'),
(97, '2025-09-05', 'vizio_com', 'P-Series', '75', 4, 'I would like to request better multi-user profile support on SmartCast and WatchFree Plus. Currently the TV treats all viewing as a single user which means recommendations and watch history are mixed together for the entire household. Adding individual user profiles with separate preferences, watch histories, and content recommendations would significantly improve the experience for families.', 'feature_request'),
(98, '2024-04-20', 'bestbuy', 'Quantum', '65', 4, 'Requesting VIZIO add support for Matter and Thread smart home protocols. As the smart home ecosystem evolves it would be valuable to have the TV serve as a Matter controller or bridge device. Being able to dim smart lights when starting a movie, adjust the thermostat during long viewing sessions, or display security camera feeds on screen without a separate app would make the VIZIO TV the true hub of the connected home.', 'feature_request'),
(99, '2025-08-18', 'reddit', 'M-Series', '55', 4, 'Please VIZIO add offline content caching for WatchFree Plus AVOD content. It would be great to be able to download select WFP movies and shows over WiFi when the connection is good and then watch them without buffering issues later. This would solve many of the streaming quality complaints about WFP and would be especially useful for users with inconsistent internet connections.', 'feature_request'),
(100, '2024-08-05', 'amazon', 'OLED', '65', 5, 'The OLED is amazing and I have one feature request. Please add pixel-level brightness compensation that adjusts over time to prevent and mitigate any potential burn-in. While OLED technology has improved dramatically in terms of burn-in resistance, having a proactive compensation algorithm that monitors pixel usage patterns and automatically adjusts to maintain uniformity would provide additional peace of mind.', 'feature_request'),
(101, '2025-04-25', 'vizio_com', 'V-Series', '65', 3, 'Please add native Bluetooth audio output to the V-Series line. The ability to connect wireless headphones directly to the TV for private listening is a basic feature that most competitors offer across their entire product range. Currently connecting Bluetooth headphones to the V-Series is not supported which is a significant limitation for households where someone wants to watch TV late at night without disturbing others.', 'feature_request'),
(102, '2025-06-28', 'bestbuy', 'P-Series', '55', 4, 'Would love to see VIZIO implement a universal search and recommendation engine that works across all installed streaming apps and WatchFree Plus simultaneously. When I search for a movie or show I want to see every available option across all my services in one unified result page with pricing and quality information. Google TV and Apple TV do this well and SmartCast should offer similar functionality.', 'feature_request'),
(103, '2024-11-15', 'twitter', 'Quantum', '55', 4, 'VIZIO should add WatchFree Plus channel recording and a pause live TV feature. Being able to pause rewind and record live WFP channels would transform the platform from a nice-to-have into a genuine cable replacement. The technology exists and other platforms offer similar DVR-like functionality for free channels. This would be a massive competitive advantage for VIZIO.', 'feature_request'),
(104, '2025-01-30', 'amazon', 'M-Series', '65', 4, 'Request for VIZIO to add customizable quick settings or a toolbar that slides in from the edge of the screen during viewing. Currently accessing picture mode, audio settings, or input selection requires navigating to the full settings menu which interrupts the viewing experience. A quick access toolbar similar to a phone notification shade would make daily usage much more efficient.', 'feature_request'),
(105, '2025-09-15', 'reddit', 'V-Series', '50', 3, 'VIZIO needs to add more international content to WatchFree Plus. The channel selection is very US-centric and there is a huge opportunity to serve multilingual households with channels and content in Spanish, Chinese, Korean, Hindi, and other languages. Adding international news channels, foreign language movies, and culturally diverse programming would expand the WFP audience significantly.', 'feature_request'),
(106, '2024-07-18', 'vizio_com', 'P-Series', '65', 4, 'I wish SmartCast had a proper widget system for the home screen. Weather, sports scores, news headlines, smart home device status, and calendar events displayed as glanceable widgets on the home screen would add practical daily utility to the TV beyond entertainment. The large screen is perfect for displaying this kind of ambient information when not actively watching content.', 'feature_request'),
(107, '2025-05-22', 'amazon', 'Quantum', '75', 4, 'Request for VIZIO to add an advanced audio processing mode that can enhance dialogue clarity without an external soundbar. Many of us watch TV with the built-in speakers and struggle to hear dialogue over music and sound effects especially in movies. Having a dedicated dialogue enhancement mode that uses AI to isolate and boost speech frequencies would be incredibly useful.', 'feature_request'),
(108, '2024-10-25', 'bestbuy', 'M-Series', '55', 3, 'Please add energy monitoring and eco-friendly features to SmartCast. I would love to see real-time power consumption data, monthly energy usage tracking, and smart suggestions for reducing power consumption without significantly impacting picture quality. An eco mode that automatically adjusts settings based on room occupancy detected through the remote or ambient sensor would be innovative.', 'feature_request'),
(109, '2025-07-30', 'reddit', 'OLED', '55', 5, 'The OLED panel is stunning and I would love VIZIO to push the envelope further with display innovation. Please consider adding support for a true 1000-nit HDR mode that unlocks the full brightness potential of the panel for limited durations during HDR highlights. Also HDR10 Plus Adaptive support alongside the existing Dolby Vision would give users the widest possible HDR compatibility.', 'feature_request'),
(110, '2025-02-10', 'twitter', 'V-Series', '43', 3, 'VIZIO should add a proper companion app that works with WatchFree Plus for second screen experiences. Being able to browse the WFP channel guide on my phone, queue up content, and send it to the TV would be much easier than navigating with the remote. Also adding social viewing features where friends can sync playback and chat while watching the same WFP content remotely would add a unique social dimension.', 'feature_request');

In [ ]:
%%sql -r fb_insert_06
-- CUSTOMER_FEEDBACK: Mixed reviews and remaining rows (111-150)
INSERT INTO VIZIO_ANALYTICS_LAB.AI_OBJECTS.CUSTOMER_FEEDBACK VALUES
(111, '2024-03-28', 'amazon', 'M-Series', '55', 3, 'The M-Series 55 is a study in contradictions. On one hand the picture quality is genuinely good for the price. Colors are accurate, HDR content looks dynamic, and the panel brightness is sufficient for most viewing environments. On the other hand the smart TV platform is a mixed bag. Some days SmartCast runs smoothly and WatchFree Plus channels load quickly. Other days the system is sluggish, apps crash, and WFP buffers endlessly. The inconsistency is what frustrates me most. Average TV with above-average picture quality and below-average software reliability.', 'review'),
(112, '2024-06-10', 'bestbuy', 'P-Series', '55', 3, 'The P-Series delivers outstanding picture quality that is undermined by an underwhelming audio experience. The display is beautiful with excellent HDR, vibrant colors, and smooth motion handling. However the built-in speakers are thin and lacking in bass producing sound that does not match the visual quality. I know a soundbar solves this but a TV at this price should have better integrated audio. The SmartCast platform works adequately most of the time but has occasional hiccups. WatchFree Plus is a nice bonus feature that I use daily for background content.', 'review'),
(113, '2024-09-22', 'reddit', 'Quantum', '55', 3, 'Three months with the Quantum 55 and my feelings are mixed. The quantum dot color technology is legitimately impressive and produces images that are noticeably more vibrant than standard LED TVs. Gaming performance is solid with good input lag and VRR support. However the SmartCast software has been inconsistent with periodic slowdowns and the occasional crash. WatchFree Plus works well for the most part but the channel selection could be broader. The remote control quality feels cheap compared to the TV itself.', 'review'),
(114, '2025-01-18', 'vizio_com', 'V-Series', '65', 3, 'The V-Series 65 offers a big screen at a small price which is its primary appeal. The picture quality is acceptable for casual viewing with decent brightness and serviceable color reproduction. It is not going to impress videophiles but for watching sports, news, and casual streaming it gets the job done. SmartCast works but is noticeably slower than on higher-end VIZIO models. WatchFree Plus loads reliably most of the time. WiFi connectivity has been mostly stable with occasional dropouts during peak usage hours.', 'review'),
(115, '2025-04-02', 'amazon', 'OLED', '55', 4, 'The VIZIO OLED delivers incredible picture quality that is only slightly marred by a few software quirks. The blacks are perfect, the colors are stunning, and HDR content looks absolutely breathtaking. However SmartCast on this model occasionally freezes requiring a restart and the boot time after a restart is longer than I would like. WatchFree Plus works well on the OLED and the content looks surprisingly good despite most channels not being in 4K resolution. Minor software issues aside this is a remarkable television.', 'review'),
(116, '2024-08-08', 'bestbuy', 'M-Series', '50', 3, 'Middle of the road TV that does not excel in any particular area but does not have any catastrophic flaws either. The M-Series 50 provides a competent viewing experience with decent picture quality, functional smart features, and adequate build quality. WatchFree Plus adds some value with its free channel lineup. The Dolby Vision support is nice but the peak brightness limits the HDR impact compared to brighter competitors. This is the definition of an average TV that will satisfy buyers with moderate expectations.', 'review'),
(117, '2025-06-08', 'reddit', 'V-Series', '43', 3, 'The V-Series 43 is fine. Not great, not terrible, just fine. It turns on, shows a picture, and gives you access to streaming apps and WatchFree Plus. The picture is adequate for a bedroom or kitchen with decent sharpness and acceptable colors. Smart features work most of the time with occasional slowness. WiFi stays connected more often than not. The remote is basic but functional. For the price you get what you pay for which is a functional television without any premium frills.', 'review'),
(118, '2024-11-28', 'amazon', 'P-Series', '65', 4, 'The P-Series 65 is nearly a perfect TV held back by software inconsistencies. The display itself is gorgeous with excellent local dimming, vivid quantum colors, and smooth motion handling for sports. Movies in Dolby Vision look reference-quality in a dark room. Gaming is responsive and visually stunning. Then you bump into SmartCast and the magic fades a little. The interface is functional but not as polished as competing platforms. Ninety percent of the time this is a five star TV. That remaining ten percent of software friction costs it one star.', 'review'),
(119, '2025-08-12', 'vizio_com', 'Quantum', '50', 3, 'Mixed experience with the Quantum 50. The color performance is excellent as expected from quantum dot technology. Movies and shows look vibrant and lifelike. However I have had intermittent issues with the HDMI ports not recognizing connected devices on first power-up. WatchFree Plus provides good free content but occasionally has audio sync issues on certain channels. The TV runs a bit warm which makes me wonder about long-term reliability.', 'review'),
(120, '2025-03-05', 'bestbuy', 'M-Series', '65', 3, 'The M-Series 65 is a decent large-screen option that comes with some caveats. Picture quality is solid for the price with good color and reasonable HDR performance. The large screen enhances movies and sports significantly. However the panel uniformity on my unit is not perfect with some visible banding in gradient scenes. SmartCast runs adequately but I notice slowdowns after the TV has been on for extended periods. WatchFree Plus is my go-to for casual viewing and it works reliably.', 'review'),
(121, '2024-04-18', 'amazon', 'V-Series', '55', 4, 'I have to say the V-Series 55 has been a pleasant surprise for our playroom TV. The kids watch cartoons on WatchFree Plus and various streaming apps all day and the TV has handled it like a champ. Picture quality is more than adequate for animated content and the colors are cheerful and bright. The smart platform is not the fastest but the kids do not care about load times. A solid budget choice for a household with kids who need a dedicated TV for their content.', 'review'),
(122, '2025-05-15', 'twitter', 'P-Series', '75', 4, 'The P-Series 75 is a magnificent display slightly held back by its software. Movie nights are incredible with this screen. The HDR performance and color accuracy make every film feel cinematic. We hosted a Super Bowl party and the TV was the star of the show. WFP sports channels looked great on the big screen. Minor software glitches here and there but nothing that ruins the experience. Worth the investment for the picture quality alone.', 'review'),
(123, '2024-07-30', 'amazon', 'Quantum', '55', 3, 'Bought the Quantum 55 expecting premium performance and got a mixed result. Colors are undeniably beautiful thanks to the quantum dot technology. Gaming performance is good with responsive inputs and smooth frame rates. However the TV exhibits some backlight blooming in dark scenes that is distracting for movie viewing. The SmartCast platform is adequate but not exceptional. WatchFree Plus channels work fine but the selection needs expansion.', 'review'),
(124, '2025-07-15', 'bestbuy', 'OLED', '65', 4, 'The VIZIO OLED is phenomenal for movies and drama. The perfect blacks and infinite contrast create images that are simply breathtaking in a dark room. Colors are rich and accurate. However I have noticed the panel brightness is lower than some competing OLEDs which can be an issue in brighter rooms during daytime viewing. SmartCast works well on this model with WatchFree Plus providing good supplementary content. A fantastic TV for dark-room movie viewing with some compromises in bright-room performance.', 'review'),
(125, '2024-10-08', 'reddit', 'M-Series', '43', 3, 'Adequate small TV for a guest room. The M-Series 43 provides a reasonable picture with Dolby Vision support which is nice for a smaller display. SmartCast performs well enough for occasional use. WatchFree Plus gives guests access to free entertainment without needing to share streaming passwords. A perfectly average TV that serves its purpose as a secondary display without generating strong feelings in either direction.', 'review'),
(126, '2025-02-05', 'amazon', 'V-Series', '50', 3, 'The V-Series 50 is a reasonable purchase if you set your expectations at the right level. For the price you get a functional 4K TV with smart features, WatchFree Plus access, and a picture that is acceptable for everyday viewing. You do not get premium HDR performance, wide viewing angles, or a blazing fast smart platform. But those trade-offs are expected at this price point. The WiFi connection has been more stable than reviews led me to expect though I am using a mesh network.', 'review'),
(127, '2025-09-20', 'vizio_com', 'P-Series', '55', 4, 'Nine months with the P-Series 55 and overall I am satisfied. The picture quality remains excellent and is the primary reason I chose this TV. Local dimming, color accuracy, and HDR performance are all above average for the price. SmartCast has been stable with only a couple of freezes over the entire ownership period. WatchFree Plus has become a daily habit for morning news and evening background entertainment. My main wish is for a faster SmartCast boot time but otherwise this is a TV I am happy to recommend.', 'review'),
(128, '2024-05-25', 'bestbuy', 'Quantum', '65', 4, 'The Quantum 65 delivers a premium viewing experience at a moderate price. The quantum dot display produces images that are richly colored and impressively bright. Sports, movies, and gaming all look great on this TV. The local dimming does a commendable job of maintaining contrast though it is not perfect with some visible blooming in challenging scenes. SmartCast runs smoothly for the most part and WatchFree Plus has become a go-to content source for our family. A good TV that earns a solid recommendation.', 'review'),
(129, '2025-06-22', 'amazon', 'M-Series', '55', 3, 'Six months in and the M-Series 55 is a perfectly adequate television. It does not wow me with its picture quality or frustrate me with major issues. The Dolby Vision support is appreciated and streaming content looks clean and detailed. WatchFree Plus provides enough free content to keep it interesting between streaming binges. The remote works reliably and the smart platform is stable if not particularly speedy. I dock two stars for the middling HDR brightness and occasional slow response from SmartCast.', 'review'),
(130, '2024-09-08', 'twitter', 'V-Series', '43', 4, 'Picked up the V-Series 43 for my dorm room and it is exactly what a college student needs. Affordable, good enough picture for Netflix and gaming, and WatchFree Plus means I have free entertainment when I cannot afford streaming subs. SmartCast covers all the apps I use and the TV is compact enough to fit on my desk. No complaints for the price. Great value pick for budget-conscious buyers.', 'review'),
(131, '2025-08-30', 'amazon', 'P-Series', '65', 5, 'One year with the P-Series 65 and it continues to be the best TV I have owned. The picture quality is outstanding with rich colors, deep blacks from the local dimming, and excellent HDR pop. Every firmware update has brought improvements and the SmartCast platform is noticeably faster than when I first bought the TV. WatchFree Plus keeps expanding its channel lineup and has become our default background entertainment. Reliability has been excellent with zero hardware issues over twelve months of daily use.', 'praise'),
(132, '2024-08-18', 'vizio_com', 'OLED', '55', 5, 'The VIZIO OLED continues to be the best value in the premium TV segment. After eight months of daily use the picture quality remains as stunning as day one with no visible image retention or burn-in issues. SmartCast has been reliable on this model and WatchFree Plus provides excellent supplementary content. Cannot recommend this TV highly enough for anyone who prioritizes picture quality.', 'praise'),
(133, '2025-04-30', 'bestbuy', 'Quantum', '55', 5, 'The Quantum 55 inch has been fantastic. Brilliant quantum dot colors, smooth gaming performance, and a reliable smart TV experience. WatchFree Plus is loaded with content that I actually want to watch. Picture quality exceeds what I expected at this price. VIZIO has earned a loyal customer with this product.', 'praise'),
(134, '2024-11-25', 'amazon', 'M-Series', '50', 4, 'After careful comparison shopping I chose the M-Series 50 and have been mostly happy with the decision. Picture quality is above average with good Dolby Vision performance. The SmartCast platform is stable and WatchFree Plus adds genuine value. Sound quality from the built-in speakers is the main weakness but a budget soundbar solved that nicely. For the overall package and price this TV represents strong value.', 'review'),
(135, '2025-03-25', 'reddit', 'V-Series', '55', 2, 'The green tint issue on my V-Series 55 is unbearable. The entire screen has a noticeable green cast that affects all content. White backgrounds look greenish, skin tones look sickly, and the overall color balance is completely off. I have tried every picture mode and color setting adjustment available but cannot eliminate the green tint. This is clearly a panel defect rather than a settings issue. Currently working with VIZIO support on a warranty claim but the process has been slow.', 'complaint'),
(136, '2025-01-25', 'amazon', 'V-Series', '32', 3, 'The V-Series 32 is an okay TV for what it is. Serviceable picture quality, basic smart features, and WatchFree Plus access. The size is right for a small room but the picture does not hold up well when you start looking for detail or color accuracy. Fine for a third or fourth TV in the house but I would not want it as my primary display.', 'review'),
(137, '2024-06-05', 'bestbuy', 'P-Series', '75', 5, 'The P-Series 75 is magnificent. The combination of massive screen size and excellent picture quality creates an experience that simply cannot be matched by smaller TVs. The local dimming is sophisticated enough to provide OLED-like contrast in dark scenes while maintaining the LED brightness advantage in bright content. Sports on this TV are an event. Movie nights feel like a trip to the cinema. WatchFree Plus on a 75 inch screen is a genuinely different experience. VIZIO delivered the ultimate home theater TV.', 'praise'),
(138, '2025-09-25', 'vizio_com', 'M-Series', '65', 4, 'The M-Series 65 has been our main family TV for ten months and it has served us well. The large screen is perfect for family viewing and the picture quality satisfies both the casual viewers and the more discerning eyes in the household. Dolby Vision content looks impressive. WatchFree Plus has earned a permanent spot in our daily viewing rotation. SmartCast is reliable enough for daily use with only rare hiccups.', 'review'),
(139, '2024-10-15', 'amazon', 'Quantum', '50', 4, 'The Quantum 50 is a compact powerhouse. The quantum dot display in this smaller size creates a dense vibrant picture that is perfect for close viewing distances. Colors are excellent and the brightness levels make HDR content impactful. Gaming at this size with the low input lag is a great experience. WatchFree Plus runs smoothly. A strong recommendation for anyone wanting premium picture quality in a more manageable screen size.', 'review'),
(140, '2025-05-30', 'bestbuy', 'V-Series', '50', 2, 'Regret this purchase. The V-Series 50 has been a constant source of frustration. WiFi drops daily requiring manual reconnection. SmartCast is painfully slow. WatchFree Plus buffers on what should be a more than adequate internet connection. The picture quality is mediocre with washed out colors and poor black levels. The only positive is that the TV was cheap but as they say you get what you pay for. Saving up for a better TV.', 'complaint'),
(141, '2024-12-10', 'reddit', 'OLED', '65', 5, 'Still in awe of this OLED every time I turn it on. The picture quality is in a completely different league. Watched a space documentary last night and the contrast between the stars and the void of space was absolutely breathtaking. No LED TV can replicate what OLED does with darkness. WatchFree Plus even looks good on this panel thanks to excellent upscaling. This is the TV that made me understand what people mean when a display is transformative.', 'praise'),
(142, '2025-07-12', 'amazon', 'P-Series', '55', 5, 'Fifteen months of ownership and the P-Series 55 remains excellent. Firmware updates have consistently improved performance. The picture is still vibrant and accurate. SmartCast has become more stable with each update. WatchFree Plus continues to add new channels and content. Reliability has been perfect with zero issues. This TV has made me a VIZIO loyalist.', 'praise'),
(143, '2024-05-12', 'vizio_com', 'M-Series', '55', 3, 'Decent TV held back by some annoying issues. The picture is good enough for daily viewing with nice Dolby Vision support. SmartCast works but is slow to load. WatchFree Plus is a nice free bonus. However the TV occasionally produces a high-pitched whine from the panel that is audible during quiet scenes. Also the CEC implementation is unreliable. These are not dealbreaker issues but they add up to a less than ideal daily experience.', 'review'),
(144, '2025-08-05', 'bestbuy', 'Quantum', '65', 5, 'This Quantum 65 is the centerpiece of our entertainment setup and it delivers every single day. The quantum dot colors are jaw-dropping and the brightness makes HDR content truly spectacular. We use it for everything from movies to gaming to WatchFree Plus and it handles all content beautifully. SmartCast has been stable and responsive. VIZIO hit a home run with the Quantum lineup.', 'praise'),
(145, '2024-09-28', 'twitter', 'V-Series', '43', 2, 'Major disappointment. Bought the V-Series 43 for my parents and they call me every week with a new problem. WiFi drops, apps crash, remote does not respond, WatchFree Plus shows error messages. I have done three factory resets and multiple firmware updates but the problems keep coming back. Should not have to provide ongoing tech support for a TV purchase.', 'complaint'),
(146, '2025-06-02', 'amazon', 'P-Series', '65', 4, 'The P-Series 65 delivers exceptional picture quality that is only slightly held back by minor software friction. The display technology is genuinely impressive with local dimming that creates dramatic contrast and colors that are both vibrant and accurate. WatchFree Plus provides a nice complement to paid streaming services. SmartCast has improved with updates though it is still not as polished as some competitors. A strong buy for picture quality enthusiasts.', 'review'),
(147, '2024-07-12', 'reddit', 'M-Series', '50', 4, 'The M-Series 50 is my recommendation for anyone asking me for a TV suggestion. It offers the best balance of price, picture quality, features, and reliability in the mid-range segment. Dolby Vision looks great, the smart platform covers all major apps, WatchFree Plus adds free content value. Sometimes the best product is the one that does everything well rather than one thing spectacularly.', 'review'),
(148, '2025-09-08', 'vizio_com', 'OLED', '55', 5, 'Approaching one year with the OLED 55 and it remains the single best electronics purchase I have ever made. The picture quality continues to astound me on a daily basis. Every movie every show every game looks better on this TV than on any display I have previously owned. WatchFree Plus has grown into a genuinely useful content platform. Zero regrets and I would buy it again at full price in a heartbeat.', 'praise'),
(149, '2025-02-20', 'bestbuy', 'V-Series', '55', 2, 'The screen uniformity on my V-Series 55 is terrible. Visible clouding in the corners and flashlighting along the edges make dark content distracting to watch. Combined with the below-average black levels and limited contrast this TV struggles with any content that has dark scenes. Movies with letterboxing are particularly bad as the black bars are visibly grey and uneven. Quality control needs to be better.', 'complaint'),
(150, '2024-08-02', 'amazon', 'Quantum', '55', 4, 'Really enjoying the Quantum 55 overall. The quantum dot color technology is the standout feature producing images that are richly saturated and incredibly lifelike. Gaming with HDR enabled is a visual treat and the low input lag keeps competitive gaming responsive. WatchFree Plus has been a great discovery with more free content than I expected. SmartCast is functional though I wish it was a bit faster. Minor complaints in what is otherwise a very satisfying TV purchase.', 'review');

In [ ]:
%%sql -r step7b_verify_feedback
-- CUSTOMER_FEEDBACK: Sample of 150 product reviews
-- Distribution: ~21% praise (rating 5), ~29% complaints (rating 1-2), ~20% feature_request (rating 3-4), ~29% review (rating 3-4)
-- Sources: amazon, bestbuy, vizio_com, reddit, twitter
-- Full 150 rows were loaded via the lab setup session. Run this cell to verify or re-populate a sample.
-- Showing representative sample INSERT (first 10 rows of each type):

-- Verify existing data
SELECT FEEDBACK_TYPE, COUNT(*) AS CNT, ROUND(AVG(RATING),1) AS AVG_RATING
FROM VIZIO_ANALYTICS_LAB.AI_OBJECTS.CUSTOMER_FEEDBACK
GROUP BY FEEDBACK_TYPE
ORDER BY CNT DESC;

In [ ]:
%%sql -r kb_insert_01
-- VIZIO_KNOWLEDGE_BASE: Policy/FAQ docs (rows 1-20)
INSERT INTO VIZIO_ANALYTICS_LAB.AI_OBJECTS.VIZIO_KNOWLEDGE_BASE VALUES
(1, 'WatchFree+ Channel Onboarding Process', 'WatchFree+ (WFP) is VIZIO''s free ad-supported streaming television (FAST) service available on all SmartCast-enabled VIZIO and ONN TVs. The channel onboarding process for content partners involves several key steps. First, prospective channel partners must submit an application through the VIZIO Partner Portal providing details about their content catalog, target audience demographics, and advertising capabilities. Applications are reviewed by the WFP Content Acquisition team within 15 business days. Approved partners receive access to the WFP Technical Integration Kit which includes specifications for HLS/DASH stream delivery, EPG metadata formatting requirements, and ad insertion protocols using SCTE-35 markers for server-side ad insertion (SSAI). Content must meet VIZIO quality standards including minimum 720p resolution for live channels and 1080p for on-demand AVOD content. All content must include proper closed captioning and content ratings metadata. The technical integration process typically takes 4-6 weeks and includes stream quality validation, ad insertion testing, and EPG accuracy verification. Once approved, channels are assigned an EPG position based on genre category and projected viewership. WFP currently supports over 300 live channels and 10,000+ on-demand titles across categories including News, Sports, Entertainment, Lifestyle, Kids, and Movies.', 'watchfree_plus'),
(2, 'WatchFree+ Content Categories and Curation Guidelines', 'WatchFree+ organizes content into primary categories to help users discover programming that matches their interests. The main categories include News and Weather, Sports, Entertainment, Movies, Lifestyle, Kids and Family, and Music. Content curation follows a data-driven approach that combines viewership analytics, seasonal relevance, and editorial recommendations. The WFP Content team reviews channel performance metrics weekly including unique viewers, average session duration, completion rates, and ad fill rates. Channels consistently performing below minimum thresholds receive optimization recommendations or may be repositioned in the EPG. Featured content on the WFP home screen is selected based on a combination of algorithmic popularity signals and editorial picks, with refresh cycles occurring every Tuesday and Friday. Content partners are encouraged to provide promotional assets and metadata updates at least two weeks before major programming events.', 'watchfree_plus'),
(3, 'WatchFree+ Revenue Model and Advertising Framework', 'WatchFree+ operates on an advertising-supported model where revenue is generated through video advertisements served during content playback. The advertising framework supports multiple ad formats including pre-roll, mid-roll, and interactive overlays. Ad serving is managed through VIZIO''s programmatic advertising platform which integrates with major demand-side platforms (DSPs) and supports both programmatic guaranteed and real-time bidding (RTB) transactions. Revenue sharing with content partners follows a tiered structure based on content type and performance metrics. VIZIO''s Inscape ACR technology provides advertisers with advanced targeting capabilities including viewing behavior analysis, content affinity mapping, and cross-platform measurement. All advertising must comply with VIZIO''s Ad Policy. The minimum ad load for WFP channels is 8 minutes per hour with a maximum of 16 minutes per hour.', 'watchfree_plus'),
(4, 'SmartCast OS Version History and Feature Roadmap', 'SmartCast is VIZIO''s proprietary smart TV operating system that powers the user interface and application ecosystem across all VIZIO and partner OEM television products. SmartCast 5.0 introduced the redesigned home screen with content-forward recommendations and integrated WatchFree+ access. SmartCast 5.1 added ProGaming Engine support with auto-detection of game consoles. SmartCast 5.2 brought enhanced voice control with expanded natural language processing. SmartCast 6.0 represented a major architectural overhaul with improved app launch times by forty percent, HDR10+ Adaptive support, and Apple HomeKit/AirPlay 2 integration. SmartCast 6.1 added multi-user profile support with personalized content recommendations. The current release SmartCast 6.2 focuses on performance optimization, improved Bluetooth audio codec support including aptX and LC3, and enhanced AirPlay 2 reliability. All SmartCast updates are distributed over-the-air (OTA) and install automatically during standby periods.', 'smartcast_platform'),
(5, 'SmartCast Home Screen Layout and Navigation Architecture', 'The SmartCast home screen serves as the primary navigation hub for all TV functions and content discovery. The layout is designed around a content-first philosophy that prioritizes recommendations and quick access to frequently used apps and services. The top navigation bar provides access to Home, WatchFree+, Apps, Inputs, and Settings. The Home section features a hero banner highlighting trending content followed by horizontally scrolling content rails organized by category including Continue Watching, Trending Now, Recommended for You, and genre-specific rails. The WatchFree+ section provides direct access to the FAST channel guide and AVOD content library. The Apps section displays installed applications in a customizable grid layout. The Inputs section shows all connected HDMI and other input sources with automatic device detection using CEC device names. Quick Settings can be accessed by pressing the gear button on the remote.', 'smartcast_platform'),
(6, 'Firmware Update Cadence and Distribution Policy', 'VIZIO maintains a regular firmware update schedule to ensure all television products receive security patches, feature improvements, and bug fixes throughout their supported lifecycle. Critical Security Updates are deployed within 72 hours of vulnerability identification and are mandatory. Feature Updates are released quarterly and go through a phased rollout starting with five percent of eligible devices in week one, expanding to twenty-five percent in week two, and reaching full deployment by week four. Maintenance Updates are released monthly and address minor bugs, app compatibility updates, and incremental performance improvements. All firmware updates are distributed over-the-air via the TV''s internet connection and are installed automatically during standby periods by default. Users can check for updates manually through Settings then System then Check for Updates. VIZIO provides firmware support for a minimum of four years from the date of product launch with extended support available for premium product lines.', 'firmware'),
(7, 'Firmware Rollback and Recovery Procedures', 'In the event that a firmware update causes functionality issues, VIZIO provides several recovery mechanisms. The standard troubleshooting sequence begins with a soft reset (holding the power button for 10 seconds). If issues persist, a factory reset through Settings then System then Reset and Admin will restore the TV to its original firmware state. For persistent issues, VIZIO Support can initiate a remote firmware rollback to the previous stable version through the VIZIO Diagnostics Protocol. In rare cases where the TV becomes unresponsive, a USB recovery method is available using recovery firmware downloaded from the VIZIO Support website loaded onto a FAT32-formatted USB drive. VIZIO tracks firmware-related issues through automated telemetry data and proactively pauses update rollouts if error rates exceed acceptable thresholds during phased deployment.', 'firmware'),
(8, 'App Certification and SmartCast App Store Requirements', 'All applications distributed through the SmartCast platform must complete VIZIO''s App Certification Program. Functional testing verifies that all app features work correctly across supported TV models and SmartCast versions. Performance benchmarking measures app launch time (must be under 5 seconds on reference hardware), memory consumption (must not exceed 350MB), and frame rate consistency (minimum 30fps for UI, 60fps for video). Security review includes static code analysis, network traffic inspection for encrypted communications, and privacy compliance verification. User experience compliance ensures apps follow SmartCast design guidelines including consistent navigation patterns and proper back button behavior. The full certification process takes 10-15 business days and apps must be recertified for each major SmartCast version update.', 'app_ecosystem'),
(9, 'App Ecosystem Partner Integration Guidelines', 'VIZIO''s app ecosystem supports a wide range of applications through the SmartCast platform. Supported development frameworks include HTML5/JavaScript for web-based apps, native C++ for performance-critical applications, and React Native for cross-platform development. All apps must implement the SmartCast Deep Link Protocol which enables universal search results to launch directly into specific content. The Chromecast Built-in integration allows mobile apps to cast content to the TV using the Google Cast SDK. VIZIO provides developer tools including a SmartCast emulator for desktop testing, remote debugging via Chrome DevTools Protocol, and automated testing frameworks for CI/CD pipeline integration. App updates are managed through the SmartCast App Distribution Platform which supports staged rollouts, A/B testing, and automatic rollback capabilities.', 'app_ecosystem'),
(10, 'Device Warranty Terms and Coverage Details', 'VIZIO provides comprehensive warranty coverage for all television products. V-Series televisions include a one-year limited warranty. M-Series and Quantum Series include a one-year limited warranty with option to purchase extended coverage for up to three additional years. P-Series Quantum include a two-year limited warranty. OLED televisions include a three-year limited warranty that specifically covers OLED panel burn-in for the first two years. For televisions 42 inches and smaller, warranty service is provided through advance exchange. For televisions larger than 42 inches, in-home service is provided by VIZIO''s authorized service network. The warranty does not cover damage from accidents, misuse, unauthorized modifications, power surges, or cosmetic damage. Proof of purchase from an authorized retailer is required for all warranty claims.', 'device_specs'),
(11, 'TV Product Line Specifications and Positioning', 'VIZIO''s television portfolio consists of five distinct product lines. The V-Series is the entry-level offering with full-array LED backlighting, 4K UHD resolution on 40-inch and larger, HDR10 support, and SmartCast with WatchFree+. The M-Series is the mid-range offering with Dolby Vision HDR, full-array LED with local dimming zones, wider color gamut, and HDMI 2.1. The M-Series Quantum combines M-Series features with quantum dot color technology. The P-Series Quantum is the premium LED offering with highest zone count local dimming, ProGaming Engine with 4K 120Hz and VRR, and widest color gamut. The OLED line represents the pinnacle with self-emissive OLED panels, perfect blacks, infinite contrast, and wide viewing angles. Available in 55 and 65 inch sizes.', 'device_specs'),
(12, 'TV Display Technology Comparison and Buying Guide', 'Choosing the right VIZIO TV depends on viewing environment, primary use case, and budget. For bright rooms, the P-Series Quantum and Quantum models offer high peak brightness exceeding 1000 nits with effective anti-reflection coatings. For dedicated home theaters, the OLED series delivers perfect blacks and infinite contrast. For gaming, the P-Series Quantum offers 4K 120Hz, VRR, ALLM, and ProGaming Engine. The OLED also excels for gaming with near-instantaneous pixel response times. For general everyday viewing, the M-Series provides excellent balance with Dolby Vision and local dimming. The V-Series is ideal for secondary rooms, bedrooms, and kitchens where reliable smart TV functionality at an affordable price is the priority.', 'device_specs'),
(13, 'ACR Technology Overview and Privacy Controls', 'Automatic Content Recognition (ACR) is a technology used by VIZIO televisions to identify content being displayed on the screen through audio and visual fingerprinting. VIZIO''s ACR implementation, marketed as Inscape, captures small samples at regular intervals and compares them against a reference database. ACR data collection requires explicit user consent during initial TV setup. Users can change their preference at any time through Settings then System then Reset and Admin then Viewing Data. Disabling ACR stops all content recognition data collection while maintaining full TV functionality including WatchFree+ access. VIZIO does not collect personally identifiable information through ACR. The data is associated with a device identifier rather than individual user identity. VIZIO publishes a transparency report annually.', 'privacy'),
(14, 'User Data Collection Practices and Privacy Framework', 'VIZIO''s Privacy Framework categorizes data collection into three tiers. Tier 1 Essential Data includes firmware version, network connectivity status, error logs, and crash reports collected by default for product functionality. Tier 2 Usage Analytics includes app usage patterns, search queries, and feature utilization metrics. Users can opt out through Settings then Privacy then Usage Analytics. Tier 3 Viewing Data includes ACR-derived content recognition data requiring explicit opt-in consent. VIZIO complies with CCPA, GDPR for European users, and COPPA for households with children under 13. Users can request data copies or deletion through the VIZIO Privacy Center at privacy.vizio.com.', 'privacy'),
(15, 'Homescreen Curation and Content Promotion Guidelines', 'The SmartCast home screen content curation combines algorithmic recommendations with editorial oversight. Recency accounts for thirty percent of ranking signal, popularity contributes twenty-five percent, personalization signals twenty-five percent, and editorial curation twenty percent. WatchFree+ content receives dedicated promotional placement including a persistent WFP rail and rotating hero banner slots. Content partners can submit promotional requests for tentpole events, new channel launches, and seasonal campaigns. The home screen Trending section updates every four hours based on real-time viewership data across the entire VIZIO fleet. All promotional content must comply with VIZIO content guidelines and pass editorial review.', 'smartcast_platform'),
(16, 'SmartCast Platform Performance Requirements and Monitoring', 'VIZIO maintains strict performance requirements for SmartCast. System boot time must not exceed 30 seconds on current-generation models and 45 seconds on previous-generation. App launch time must be under 5 seconds for first-party apps and under 8 seconds for third-party. UI frame rate must maintain minimum 30fps with target of 60fps on premium models. Memory utilization must remain below 85 percent. Network stack requires DNS resolution within 500ms and HTTP response initiation within 2 seconds. System health is monitored through the VIZIO Operations Center which aggregates telemetry data from the entire TV fleet. The platform team conducts monthly performance reviews analyzing trends in boot time, app crash rates, network errors, and user-reported issues.', 'smartcast_platform'),
(17, 'OEM Partnership Program Overview', 'VIZIO''s OEM Partnership Program enables third-party TV manufacturers to license the SmartCast platform and WatchFree+ service. Current OEM partners include Walmart''s ONN brand. OEM partners receive the full SmartCast software stack including home screen, app ecosystem, WatchFree+, and voice control. Customization options are limited to boot logos, default settings, and partner-specific app preloading. Hardware requirements include minimum quad-core ARM processor with 2GB RAM, WiFi 5 minimum with WiFi 6 recommended. OEM partners participate in SmartCast advertising revenue through negotiated share of WatchFree+ and home screen advertising revenue. The program includes technical support, integration engineering assistance, and access to the SmartCast testing lab for hardware certification.', 'smartcast_platform'),
(18, 'OEM Hardware Certification and Quality Standards', 'All OEM partner devices must pass VIZIO''s Hardware Certification Program. Display certification requires minimum 85 percent DCI-P3 color gamut for mid-range and premium, 72 percent BT.709 for budget. Peak brightness must exceed 250 nits SDR, 400 nits for HDR-capable. Thermal certification requires surface temperature not to exceed 45 degrees Celsius during sustained operation. WiFi certification requires stable connectivity at 30 feet with throughput sufficient for 4K HDR streaming at 25 Mbps. All OEM devices must pass a 500-hour accelerated life test simulating two years of typical usage. Devices that pass receive the SmartCast Certified badge and are eligible for firmware update distribution.', 'smartcast_platform'),
(19, 'Troubleshooting WiFi Connectivity Issues', 'WiFi connectivity problems are among the most common support requests. If TV cannot find WiFi network: verify router is broadcasting on compatible frequency (2.4 GHz and 5 GHz supported), try 2.4 GHz first for wider compatibility. If frequent disconnections: check for WiFi interference from neighboring networks, microwave ovens, baby monitors, Bluetooth devices. Use WiFi analyzer to find least congested channel. If signal below three bars consider WiFi extender or mesh network. If slow streaming despite good signal: run built-in network speed test. Minimum recommended: 5 Mbps HD, 15 Mbps 4K, 25 Mbps 4K HDR. Clear network cache by performing soft reset (hold power 10 seconds).', 'troubleshooting'),
(20, 'Troubleshooting Audio and HDMI Connectivity', 'No sound from speakers: verify not muted, check audio output set to TV Speakers, verify correct HDMI port for ARC (typically HDMI 1). Audio out of sync: navigate to Settings Audio Lip Sync Adjustment, increase delay in 10ms increments. HDMI no picture: verify cable connected at both ends, try different port, ensure correct input selected, check Full UHD Color enabled in Settings for 4K devices, try different HDMI cable. HDMI 2.1 features require Ultra High Speed cables. CEC not working: ensure CEC enabled on both TV and device (Settings Inputs HDMI-CEC), power cycle all connected devices to reset CEC handshake.', 'troubleshooting');

In [ ]:
%%sql -r kb_insert_02
-- VIZIO_KNOWLEDGE_BASE: Policy/FAQ docs (rows 21-40)
INSERT INTO VIZIO_ANALYTICS_LAB.AI_OBJECTS.VIZIO_KNOWLEDGE_BASE VALUES
(21, 'Troubleshooting SmartCast Performance and App Issues', 'SmartCast home screen slow or unresponsive: start with soft reset (hold power 10 seconds) to clear system cache. If persists, check for firmware updates in Settings System Check for Updates. Factory reset as last resort erases all user settings and app logins. Specific app crashes: close and reopen, navigate to Settings Apps and Clear Cache/Clear Data, uninstall and reinstall from SmartCast App Store. WatchFree Plus channels not loading: check internet connectivity (minimum 5 Mbps required), clear WFP app cache. If specific channels fail while others work, the issue may be content provider side and typically resolves within 24 hours.', 'troubleshooting'),
(22, 'Troubleshooting Picture Quality and Display Issues', 'Picture too dark: check picture mode (Calibrated Dark is designed for dark rooms), adjust backlight setting, disable Ambient Light Sensor. Colors inaccurate: use Calibrated picture mode, set Color Temperature to Normal or Warm, avoid Vivid mode for color-critical viewing. Motion blur: enable Motion Control or Clear Action settings, ensure Game Mode active for gaming. Visible banding in gradients: often related to source content compression rather than panel, try same content from different source, adjust Noise Reduction and Contour Smoothing settings.', 'troubleshooting'),
(23, 'Remote Control Specifications and Troubleshooting', 'VIZIO SmartCast remotes use IR for basic functions (power, volume, mute) and Bluetooth for advanced features (voice search, keyboard, SmartCast navigation). Voice search activated by pressing and holding microphone button. If remote unresponsive: replace batteries with fresh AAA. If only Bluetooth functions affected: re-pair by holding Back and Home buttons simultaneously for 5 seconds until LED flashes, move within 3 feet of TV. If intermittent response: ensure no obstructions between remote and TV IR sensor at bottom center of display, clean IR emitter window with soft dry cloth.', 'troubleshooting'),
(24, 'Energy Efficiency and Environmental Compliance', 'All VIZIO TV models meet or exceed ENERGY STAR certification requirements. Power consumption varies by line: V-Series 30W (32") to 120W (75"), M-Series 55W (43") to 145W (75"), P-Series Quantum 85W (55") to 200W (85"), OLED 120W (55") to 150W (65"). Eco Mode reduces power consumption by approximately 30 percent. Auto-power-off feature turns TV off after 10 minutes of no signal or 4 hours of no user interaction. VIZIO participates in responsible recycling programs and all products comply with RoHS and REACH regulations.', 'device_specs'),
(25, 'HDMI Port Specifications and Input Management', 'V-Series: three HDMI 2.0 ports with one ARC. M-Series: three to four ports with at least one HDMI 2.1 and eARC. P-Series Quantum: four ports with two full HDMI 2.1 (4K 120Hz, VRR, ALLM, eARC). OLED: four HDMI 2.1 ports with full feature support on all ports. To enable advanced features: Settings Inputs select port and enable Full UHD Color (must be enabled per-port). Input management includes automatic input detection, customizable labels, and ability to hide unused inputs.', 'device_specs'),
(26, 'WatchFree+ Technical Requirements and Stream Specifications', 'WFP uses adaptive bitrate streaming via HLS protocol with quality tiers from 480p at 1.5 Mbps to 1080p at 8 Mbps. On-demand AVOD supports up to 4K UHD at 15-20 Mbps for select premium content. All content protected with Widevine L1 DRM. EPG data refreshed every 6 hours with incremental updates for last-minute changes, covering up to 72 hours of programming. WFP supports server-side ad insertion (SSAI) using SCTE-35 markers for seamless ad transitions. The client implements pre-buffering of the next channel for near-instantaneous channel switching.', 'watchfree_plus'),
(27, 'WatchFree+ Content Licensing and Partner Terms', 'WFP content is sourced through Channel License Agreements (24/7 linear channels, 1-3 year terms with auto-renewal), On-Demand Content Agreements (individual movies and series, non-exclusive, 6-24 month windows), and Content Syndication Agreements (unified distribution across live and on-demand). All content must meet minimum 720p resolution, AAC-LC audio minimum (Dolby Digital Plus preferred), complete metadata, and FCC-compliant closed captioning. Content partners have access to self-service analytics dashboard with real-time viewership, ad performance, and audience demographic insights.', 'watchfree_plus'),
(28, 'Privacy Settings Configuration and User Controls', 'Viewing Data Controls: Settings System Reset and Admin Viewing Data toggle controls ACR collection. Advertising Controls: Settings System Reset and Admin Advertising includes Limit Ad Tracking and Reset Advertising Identifier. Usage Analytics: Settings Privacy Usage Analytics governs non-essential analytics collection. Voice Data: voice queries processed in real-time and not stored by default, review and delete through Settings Privacy Voice Data. Network Data: controls sharing of WiFi network and connected device information. All settings take effect immediately and can be changed at any time without affecting TV functionality.', 'privacy'),
(29, 'SmartCast Voice Control Integration and Capabilities', 'SmartCast supports three voice interaction modes. Direct Voice via remote microphone supports content search, volume/channel control, input switching, app launching, and playback control. Google Assistant integration enables smart home device control, general knowledge queries, and cross-device casting. Amazon Alexa integration via Alexa app supports power, volume, input selection, and content launching. Apple HomeKit and Siri support allows power control, volume adjustment, and input switching. All voice interactions can be disabled through Settings Privacy Voice Control.', 'smartcast_platform'),
(30, 'Network Requirements and Connectivity Standards', 'Wireless: WiFi 5 (802.11ac) on all models, WiFi 6 (802.11ax) on P-Series Quantum and OLED. Dual-band (2.4 GHz and 5 GHz) on all models. Wired: RJ-45 Ethernet up to 100 Mbps on all models. Minimum bandwidth: 3 Mbps SD, 5 Mbps HD 720p, 10 Mbps Full HD 1080p, 15 Mbps 4K, 25 Mbps 4K HDR. All SmartCast network communications use TLS 1.2 or higher encryption. DNS fallback to Google Public DNS and Cloudflare DNS if primary server unresponsive.', 'device_specs'),
(31, 'Factory Reset Procedures and Data Retention', 'Factory reset: Settings System Reset and Admin Reset TV to Factory Defaults. Enter parental control PIN (default 0000). Data erased: all app data and logins, WiFi configs, custom picture/audio settings, SmartCast profiles, WFP favorites, paired Bluetooth devices, parental control settings, privacy preferences. Data preserved: current firmware version (no rollback), hardware calibration in NVRAM, device serial and model info. Recommend factory reset only when other troubleshooting fails due to time required to reconfigure preferences and re-enter streaming credentials.', 'troubleshooting'),
(32, 'SmartCast App Store Overview and Featured Apps', 'The SmartCast App Store hosts over 200 applications. Streaming Video: Netflix, Disney+, Max, Hulu, Peacock, Paramount+, Apple TV+, Amazon Prime Video, Tubi, Pluto TV, Plex, Crackle. Music: Spotify, Pandora, Amazon Music, Apple Music, iHeartRadio, TIDAL. Gaming: Apple Arcade and casual gaming titles. Fitness: Peloton and workout streaming. Apps download and install automatically. Users manage installed apps through Settings Apps showing storage usage per app with options to clear cache, clear data, or uninstall. SmartCast auto-updates installed apps during standby.', 'app_ecosystem'),
(33, 'Parental Control Features and Content Restrictions', 'Parental controls protected by four-digit PIN (default 0000). Content Rating Locks block content based on US TV ratings (TV-Y through TV-MA), US movie ratings (G through NC-17), and Canadian ratings. App Restrictions lock specific apps requiring PIN to launch. Input Locks prevent access to specific HDMI/other inputs without PIN. WatchFree+ Parental Controls include ability to block specific channels by genre and enable Kids Mode filtering to TV-Y through TV-G content only. Viewing Time Controls set maximum daily viewing limits per user profile. All settings accessible through Settings System Parental Controls.', 'privacy'),
(34, 'VIZIO Customer Support Channels and Service Policies', 'Phone Support: 1-844-254-8087, seven days a week, average wait under 10 minutes. Online Support: support.vizio.com with knowledge base, FAQ, firmware info, manuals, and community forum. Live Chat: available during business hours with transcripts emailed. Email Support: 24-48 business hour response time. Social Media: Twitter/X @VIZIOsupport and Facebook. Warranty claims: TVs 42" and under receive advance exchange replacement, TVs over 42" receive in-home service from authorized technicians.', 'troubleshooting'),
(35, 'Accessibility Features and Compliance', 'Visual: Text-to-Speech screen reader (Settings Accessibility Text-to-Speech), High Contrast Mode for improved readability, Zoom Mode for magnification. Hearing: comprehensive closed captioning with customizable font size, color, background, opacity, and edge style (Settings Accessibility Closed Captions). SAP support for audio description tracks. Motor: voice control for hands-free operation, USB keyboard navigation support. All features comply with FCC accessibility requirements and CVAA standards.', 'device_specs'),
(36, 'WatchFree+ International Expansion and Localization', 'WFP is primarily available in US and Canada. US market offers 300+ channels in English with growing Spanish-language selection. Canadian market operates under CRTC regulations with required Canadian content percentage, supporting English and French. Localization requirements for new markets include local content licensing, local advertising sales channels, multi-language EPG and metadata adaptation, compliance with local broadcasting regulations, and region-specific content rating systems. VIZIO works with regional content aggregators for international expansion.', 'watchfree_plus'),
(37, 'SmartCast Developer Tools and SDK Documentation', 'Primary development framework: HTML5/JavaScript adapted for TV form factor. SDK includes TV-optimized rendering engine, remote control input handling, DRM integration, and SmartCast-specific APIs. SmartCast Emulator provides desktop simulation supporting multiple resolutions, remote control simulation, network condition simulation, and performance profiling. Chrome DevTools integration for real-time debugging, DOM inspection, and JavaScript profiling. Deep Link Protocol specification defines URI scheme for launching apps to specific content. Automated testing via SmartCast Test Framework with CI/CD platform integration.', 'app_ecosystem'),
(38, 'ACR Opt-Out Process and Implications', 'To opt out: Settings System Reset and Admin Viewing Data set to Off. Takes effect immediately. When ACR disabled: content recommendations no longer reflect linear TV/antenna/external input viewing, cross-platform ad targeting stops, Viewing History stops accumulating. When ACR disabled features NOT affected: WatchFree+ fully functional, all SmartCast apps work normally, TV picture/audio/hardware features unaffected, firmware updates continue normally, app-specific recommendations unaffected. Users can request deletion of previously collected data through VIZIO Privacy Center or Support.', 'privacy'),
(39, 'Firmware Version Compatibility and Feature Matrix', 'SmartCast 4.0-4.1 (legacy): basic smart TV, WFP limited channels, HDR10 passthrough, HDMI CEC. No Dolby Vision gaming, HDMI 2.1, or multi-user profiles. SmartCast 5.0-5.2: redesigned home screen, expanded WFP, ProGaming Engine with ALLM/VRR, Google Assistant, enhanced Chromecast. Dolby Vision gaming added in 5.1 for P-Series/OLED. SmartCast 6.0: architectural overhaul with 40% faster app launch, HDR10+ Adaptive, Apple HomeKit/AirPlay 2, new app framework. SmartCast 6.1: multi-user profiles, enhanced Bluetooth codecs, 72-hour WFP EPG. SmartCast 6.2 (current): stability, audio processing, WFP optimization, security updates.', 'firmware'),
(40, 'WatchFree+ Performance Metrics and Quality Standards', 'Stream Start Time target: under 3 seconds for recent channels, under 5 seconds for new. Channels exceeding 8 seconds trigger investigation. Buffering Rate target: less than 0.5% of viewing time. Channels exceeding 1% during peak hours receive priority attention. Stream Quality target: 1080p for 70%+ of sessions, minimum 480p for 99%. Ad Delivery Rate target: 95%+ fill rate prime-time, 85% off-peak. EPG Accuracy target: 98% for next 24 hours. Channel Availability target: 99.5% monthly uptime. Outages exceeding 30 minutes trigger incident response. All metrics tracked through WFP Operations Dashboard with automated alerting.', 'watchfree_plus');

In [ ]:
%%sql -r step7c_verify_kb
-- VIZIO_KNOWLEDGE_BASE: 40 policy/FAQ documents
-- Categories: watchfree_plus (7), smartcast_platform (7), device_specs (7), troubleshooting (7), privacy (5), app_ecosystem (4), firmware (3)
-- Topics covered: WFP channel onboarding, firmware update cadence, app certification, homescreen curation,
--   ACR opt-out, device warranty, SmartCast OS versions, OEM partnership guidelines

-- Verify existing data
SELECT CATEGORY, COUNT(*) AS DOC_COUNT
FROM VIZIO_ANALYTICS_LAB.AI_OBJECTS.VIZIO_KNOWLEDGE_BASE
GROUP BY CATEGORY
ORDER BY DOC_COUNT DESC;

---
## Step 8: Final Verification
Confirm all 13 tables exist with expected row counts.

In [ ]:
%%sql -r step8_final_verification
-- Final verification: row counts for all 13 tables across all 3 databases
SELECT table_catalog AS DATABASE_NAME, table_schema AS SCHEMA_NAME, table_name AS TABLE_NAME, row_count AS ROW_COUNT
FROM VIZIO_STREAMING.INFORMATION_SCHEMA.TABLES
WHERE table_schema = 'WFP'
UNION ALL
SELECT table_catalog, table_schema, table_name, row_count
FROM VIZIO_DEVICES.INFORMATION_SCHEMA.TABLES
WHERE table_schema IN ('FLEET','ENGAGEMENT','APPS','OEM')
UNION ALL
SELECT table_catalog, table_schema, table_name, row_count
FROM VIZIO_ANALYTICS_LAB.INFORMATION_SCHEMA.TABLES
WHERE table_schema = 'AI_OBJECTS'
ORDER BY 1, 2, 3;